In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import polars as pl
import glob
import os
import gc

# 1. Configuration & Paths
BASE_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

STATIC_PATHS = {
    "accounts": f"{BASE_DIR}/accounts.parquet",
    "accounts_add": f"{BASE_DIR}/accounts-additional.parquet",
    "customers": f"{BASE_DIR}/customers.parquet",
    "linkage": f"{BASE_DIR}/customer_account_linkage.parquet",
    "products": f"{BASE_DIR}/product_details.parquet",
    "branch": f"{BASE_DIR}/branch.parquet",
    "demographics": f"{BASE_DIR}/demographics.parquet",
    "train_labels": f"{BASE_DIR}/train_labels.parquet",
    "test_accounts": f"{BASE_DIR}/test_accounts.parquet"
}

TXN_DIR = f"{BASE_DIR}/transactions"
TXN_ADD_DIR = f"{BASE_DIR}/transactions_additional"

# 2. Load Static Tables
print("--- LOADING STATIC TABLES ---")
static_dfs = {}
for name, path in STATIC_PATHS.items():
    if os.path.exists(path):
        static_dfs[name] = pl.read_parquet(path)
        print(f"{name.ljust(15)}: {static_dfs[name].shape}")
    else:
        print(f"WARNING: Missing file {path}")

# 3. Identify Transaction Shards
txn_files = glob.glob(f"{TXN_DIR}/batch-*/*.parquet")
txn_add_files = glob.glob(f"{TXN_ADD_DIR}/batch-*/*.parquet")

print("\n--- TRANSACTION SHARDS ---")
print(f"Main Transaction Files: {len(txn_files)}")
print(f"Additional Transaction Files: {len(txn_add_files)}")

# 4. Initialize the Master Account Dataframe (The Backbone)
df_train = static_dfs["train_labels"].clone()
df_test = static_dfs["test_accounts"].with_columns(pl.lit(None).alias("is_mule"))

print(f"\nColumns in train: {df_train.columns}")
print(f"Columns in test: {df_test.columns}")

# FIX: Use how="diagonal" instead of "vertical" to handle shape mismatch.
# This tells Polars to match columns by name and fill missing ones with nulls.
master_accounts = pl.concat([df_train, df_test], how="diagonal")
print(f"\nMaster Accounts (Train + Test) Backbone initialized: {master_accounts.shape}")

gc.collect()

In [ ]:
# 1. Drop Target Leakage Columns immediately
leakage_cols = ['mule_flag_date', 'alert_reason', 'flagged_by_branch']
master_accounts = master_accounts.drop([col for col in leakage_cols if col in master_accounts.columns])
print(f"Leakage columns dropped. Master shape: {master_accounts.shape}")

# 2. The Great Static Join (Polars is incredibly fast at this)
print("\n--- BUILDING STATIC FEATURE MATRIX ---")

static_features = (
    master_accounts
    .join(static_dfs["accounts"], on="account_id", how="left")
    .join(static_dfs["accounts_add"], on="account_id", how="left")
    .join(static_dfs["linkage"], on="account_id", how="left")  # Brings in customer_id
    .join(static_dfs["customers"], on="customer_id", how="left")
    .join(static_dfs["demographics"], on="customer_id", how="left")
    .join(static_dfs["products"], on="customer_id", how="left")
    .join(static_dfs["branch"], on="branch_code", how="left")
)

print(f"Final Static Matrix Shape: {static_features.shape}")

# 3. Free up RAM by deleting the individual static dataframes
del static_dfs
gc.collect()

# Let's see what columns we have to play with before hitting transactions
print("\nSample of available static columns:")
print(static_features.columns[:15])

In [ ]:
import polars as pl
import gc

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
batches = ["batch-1", "batch-2", "batch-3", "batch-4"]

batch_aggregates = []

print("--- IGNITING MANUAL MAP-REDUCE PIPELINE ---")

# STEP 1: MAP (Process one batch at a time)
for batch in batches:
    print(f"\nProcessing {batch}...")
    
    # 1. Scan only this batch
    lazy_txns = pl.scan_parquet(f"{TXN_DIR}/{batch}/*.parquet")
    
    # 2. Select ONLY the columns we need to prevent RAM bloat
    lazy_txns = lazy_txns.select([
        "account_id", "amount", "counterparty_id", "channel", "mcc_code", "txn_type"
    ])
    
    # 3. Partial Aggregation
    # Instead of counting exact uniques here, we store the unique values as a list.
    # A list of 50 unique counterparties for 160k accounts takes almost 0 RAM!
    batch_plan = (
        lazy_txns
        .group_by("account_id")
        .agg([
            pl.len().alias("tx_count"),
            pl.col("amount").sum().alias("sum_amount"),
            pl.col("amount").max().alias("max_amount"),
            
            # Keep lists of the unique values found in THIS batch
            pl.col("counterparty_id").drop_nulls().unique().alias("cp_list"),
            pl.col("channel").drop_nulls().unique().alias("ch_list"),
            pl.col("mcc_code").drop_nulls().unique().alias("mcc_list"),
            
            (pl.col("txn_type").cast(pl.String) == "CR").sum().alias("cr_count"),
            (pl.col("txn_type").cast(pl.String) == "DR").sum().alias("dr_count")
        ])
    )
    
    # 4. Execute just for this batch
    df_batch = batch_plan.collect(streaming=True)
    batch_aggregates.append(df_batch)
    print(f"{batch} completed. Intermediate shape: {df_batch.shape}")
    
    # Force garbage collection to clear RAM before the next batch
    gc.collect()


# STEP 2: REDUCE (Combine the 4 summaries)
print("\n--- COMBINING BATCHES & FINALIZING FEATURES ---")
combined = pl.concat(batch_aggregates)

final_txn_features = (
    combined.group_by("account_id")
    .agg([
        pl.col("tx_count").sum().alias("total_tx_count"),
        pl.col("sum_amount").sum().alias("total_txn_volume"),
        pl.col("max_amount").max().alias("max_txn_amount"),
        
        # We combine the lists from the 4 batches, explode them, and count the final absolute uniques
        pl.col("cp_list").explode().n_unique().alias("unique_counterparties"),
        pl.col("ch_list").explode().n_unique().alias("unique_channels"),
        pl.col("mcc_list").explode().n_unique().alias("unique_mcc_codes"),
        
        pl.col("cr_count").sum().alias("credit_count"),
        pl.col("dr_count").sum().alias("debit_count")
    ])
)

print(f"✅ Final Transaction Features Built! Shape: {final_txn_features.shape}")

# Optional: Merge it right into your static features if you still have `static_features` in memory
# master_features = static_features.join(final_txn_features, on="account_id", how="left")

In [ ]:
import polars as pl
import gc

print("--- FINALIZING MASTER MATRIX ---")

# 1. Merge the transaction features into your static features
master_features = static_features.join(final_txn_features, on="account_id", how="left")

print(f"Final Merged Master Matrix Shape: {master_features.shape}")

# 2. Fill any missing transaction data with 0 (Accounts that never made a transaction)
txn_cols = ["total_tx_count", "total_txn_volume", "max_txn_amount", 
            "unique_counterparties", "unique_channels", "unique_mcc_codes", 
            "credit_count", "debit_count"]

master_features = master_features.with_columns([
    pl.col(c).fill_null(0) for c in txn_cols
])

# 3. SAVE THE CHECKPOINT
output_path = "/kaggle/working/master_features_v1.parquet"
master_features.write_parquet(output_path)

print(f"✅ CHECKPOINT SAVED TO: {output_path}")
print("You can now safely restart your kernel anytime and just load this file!")

# Free up memory
del combined
del final_txn_features
gc.collect()

In [ ]:
import polars as pl
import glob
import gc

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
TXN_ADD_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions_additional"
batches = ["batch-1", "batch-2", "batch-3", "batch-4"]

advanced_aggregates = []

print("--- IGNITING LEVEL 2 FEATURE ENGINEERING (GEO, IP, BALANCE) ---")

for batch in batches:
    print(f"\nProcessing {batch} Join...")
    
    # 1. Load the main transactions for this batch (We just need ID and Account)
    lazy_txn = pl.scan_parquet(f"{TXN_DIR}/{batch}/*.parquet").select(["transaction_id", "account_id", "txn_type"])
    
    # 2. Load the additional transactions for this batch
    lazy_txn_add = pl.scan_parquet(f"{TXN_ADD_DIR}/{batch}/*.parquet").select([
        "transaction_id", "ip_address", "latitude", "longitude", "balance_after_transaction"
    ])
    
    # 3. JOIN THEM ON THE FLY (This maps the IP/Geo to the Account ID)
    lazy_joined = lazy_txn.join(lazy_txn_add, on="transaction_id", how="inner")
    
    # 4. Extract Elite Features (IP Entropy, Balance Drops)
    batch_plan = (
        lazy_joined
        .group_by("account_id")
        .agg([
            # Digital Identity Hopping (Mules use many IPs)
            pl.col("ip_address").drop_nulls().unique().alias("ip_list"),
            
            # Liquidity Wipeouts (Mules drain balances to exactly 0)
            pl.col("balance_after_transaction").min().alias("min_balance"),
            pl.col("balance_after_transaction").mean().alias("mean_balance")
        ])
    )
    
    # Execute batch
    df_batch = batch_plan.collect(engine="streaming")
    advanced_aggregates.append(df_batch)
    print(f"{batch} completed. Intermediate shape: {df_batch.shape}")
    
    gc.collect()

# REDUCE: Combine and Finalize
print("\n--- COMBINING BATCHES & FINALIZING LEVEL 2 FEATURES ---")
combined_adv = pl.concat(advanced_aggregates)

final_adv_features = (
    combined_adv.group_by("account_id")
    .agg([
        # Explode the IP lists from the 4 batches and get absolute unique IPs
        pl.col("ip_list").explode().n_unique().alias("unique_ips"),
        
        # Get the absolute lowest balance this account ever hit
        pl.col("min_balance").min().alias("absolute_min_balance"),
        pl.col("mean_balance").mean().alias("overall_mean_balance")
    ])
)

print(f"✅ Level 2 Features Built! Shape: {final_adv_features.shape}")

# Merge into our checkpointed Master Matrix
master_features = pl.read_parquet("/kaggle/working/master_features_v1.parquet")
master_features = master_features.join(final_adv_features, on="account_id", how="left")

# Save V2 Checkpoint!
master_features.write_parquet("/kaggle/working/master_features_v2.parquet")
print("✅ MASTER MATRIX V2 SAVED!")

del combined_adv
del final_adv_features
gc.collect()

In [ ]:
import polars as pl
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np

print("--- LOADING V2 MASTER MATRIX ---")
df_master = pl.read_parquet("/kaggle/working/master_features_v2.parquet").to_pandas()

# 1. Separate Train and Test (Phase 2 Prediction Set)
df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

print(f"Train Shape: {df_train.shape}")
print(f"Test Shape (To Predict): {df_test.shape}")

# 2. Define Features and Clean Data
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    # --- TARGET LEAKAGE COLUMNS TO DROP ---
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date'
]

features = [c for c in df_train.columns if c not in drop_cols]

# Identify Categorical Columns for CatBoost
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()

# Fill missing values: Strings with "UNKNOWN", Numbers with -1
df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

num_features = [c for c in features if c not in cat_features]
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

X = df_train[features]
y = df_train['is_mule'].astype(int)

# 3. Local Validation Split (To see how good we are)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 4. Initialize the Beast
print("\n--- TRAINING CATBOOST MODEL ---")
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    auto_class_weights='Balanced', # THIS IS CRITICAL FOR THE IMBALANCE
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100
)

# Train the model
model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

# 5. Evaluate
val_preds = model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, val_preds)

print(f"\n✅ VALIDATION ROC-AUC SCORE: {auc_score:.5f}")

# 6. Top 10 Feature Importances
importances = model.get_feature_importance()
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print("\n--- TOP 10 STRONGEST FEATURES ---")
print(feat_imp.head(10))

In [ ]:
import pandas as pd

print("--- GENERATING PREDICTIONS & FORMATTING SUBMISSION ---")

# 1. Generate the probabilities using your trained model
# We use [:, 1] to get the probability that the account IS a mule
test_probs = model.predict_proba(df_test[features])[:, 1]

# 2. Create the dataframe exactly matching the RBIH guidelines
submission = pd.DataFrame({
    'account_id': df_test['account_id'],
    'is_mule': test_probs,    # Your 0.92 AUC probabilities
    'suspicious_start': '',   # Left blank for the baseline
    'suspicious_end': ''      # Left blank for the baseline
})

# 3. Save it to Kaggle's working directory
output_csv = "/kaggle/working/DONE_WITHIT_baseline.csv"
submission.to_csv(output_csv, index=False)

print(f"✅ FORMATTED SUBMISSION SAVED TO: {output_csv}")
print("\nPreview of what you are submitting:")
print(submission.head())

In [ ]:
import polars as pl
import gc

print("--- IGNITING LEVEL 4: TOPOLOGY & STRUCTURING ---")

# 1. Load our V2 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v2.parquet")

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
batches = ["batch-1", "batch-2", "batch-3", "batch-4"]

level4_aggregates = []

# --- MAP REDUCE FOR DIRECTIONAL FLOW & ROUND AMOUNTS ---
for batch in batches:
    print(f"Processing {batch}...")
    lazy_txns = pl.scan_parquet(f"{TXN_DIR}/{batch}/*.parquet").select([
        "account_id", "counterparty_id", "txn_type", "amount"
    ])

    batch_plan = (
        lazy_txns
        .with_columns([
            # Flag mechanical round transactions (e.g., exactly 100.00, 5000.00)
            (pl.col("amount") % 100 == 0).cast(pl.Int32).alias("is_round"),
            
            # Split counterparties by Inflow (Credit) and Outflow (Debit)
            pl.when(pl.col("txn_type").cast(pl.String).str.contains("C")).then(pl.col("counterparty_id")).otherwise(None).alias("in_cpty"),
            pl.when(pl.col("txn_type").cast(pl.String).str.contains("D")).then(pl.col("counterparty_id")).otherwise(None).alias("out_cpty")
        ])
        .group_by("account_id")
        .agg([
            pl.col("is_round").sum().alias("round_tx_count"),
            pl.col("in_cpty").drop_nulls().unique().alias("in_cpty_list"),
            pl.col("out_cpty").drop_nulls().unique().alias("out_cpty_list")
        ])
    )

    df_batch = batch_plan.collect(engine="streaming")
    level4_aggregates.append(df_batch)
    gc.collect()

print("\n--- REDUCING & MERGING LEVEL 4 FEATURES ---")
combined_l4 = pl.concat(level4_aggregates)

final_l4 = (
    combined_l4.group_by("account_id")
    .agg([
        pl.col("round_tx_count").sum().alias("total_round_txns"),
        pl.col("in_cpty_list").explode().drop_nulls().n_unique().alias("in_degree"),
        pl.col("out_cpty_list").explode().drop_nulls().n_unique().alias("out_degree")
    ])
)

# Merge into Master
master_df = master_df.join(final_l4, on="account_id", how="left")

# Fill nulls with 0 for accounts with no transactions
master_df = master_df.with_columns([
    pl.col("total_round_txns").fill_null(0),
    pl.col("in_degree").fill_null(0),
    pl.col("out_degree").fill_null(0)
])

print("\n--- APPLYING PDF COMPOSITE RATIOS ---")

# 1. Round Txn Ratio (Section 4.9)
master_df = master_df.with_columns(
    (pl.col("total_round_txns") / (pl.col("total_tx_count") + 1)).alias("round_txn_ratio")
)

# 2. Degree Ratio - Network Asymmetry (Section 7.7)
master_df = master_df.with_columns(
    (pl.col("in_degree") / (pl.col("out_degree") + 1)).alias("degree_ratio")
)

# 3. Skeleton Velocity Score (Section 6.9)
master_df = master_df.with_columns(
    (pl.col("loan_count").fill_null(0) + pl.col("cc_count").fill_null(0) + pl.col("od_count").fill_null(0)).alias("total_products")
)
master_df = master_df.with_columns(
    (pl.col("total_tx_count") / (pl.col("total_products") + 1)).alias("skeleton_velocity_score")
)

# 4. Geographic Mismatch Index (Section 6.4)
# Compare the first 2 digits of PINs to see if they are from different regions entirely
master_df = master_df.with_columns([
    pl.col("customer_pin").cast(pl.String).str.slice(0, 2).alias("cust_region"),
    pl.col("branch_pin").cast(pl.String).str.slice(0, 2).alias("branch_region")
])
master_df = master_df.with_columns(
    (pl.col("cust_region") != pl.col("branch_region")).cast(pl.Int32).alias("is_remote_account")
)

# SAVE THE V3 CHECKPOINT
master_df.write_parquet("/kaggle/working/master_features_v3.parquet")
print(f"✅ V3 MASTER MATRIX SAVED! Shape: {master_df.shape}")

# Cleanup
del combined_l4, final_l4, level4_aggregates
gc.collect()

In [ ]:
import polars as pl
import gc

print("--- IGNITING LEVEL 5: BULLETPROOF ACCOUNT CHUNKING (V2) ---")

# 1. Get the list of all accounts from our V3 matrix
master_df = pl.read_parquet("/kaggle/working/master_features_v3.parquet")
all_accounts = master_df["account_id"].to_list()

# 2. Split accounts into 4 manageable chunks (~40k accounts each)
chunk_size = len(all_accounts) // 4 + 1
account_chunks = [all_accounts[i:i + chunk_size] for i in range(0, len(all_accounts), chunk_size)]

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
temporal_aggregates = []

# 3. Process Chunk by Chunk
for i, acc_chunk in enumerate(account_chunks):
    print(f"\nProcessing Account Chunk {i+1}/4 ({len(acc_chunk)} accounts)...")
    
    # Lazy scan, but FILTER for only the accounts in this chunk before loading into RAM!
    lazy_txns = (
        pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
        .select(["account_id", "transaction_timestamp"])
        .filter(pl.col("account_id").is_in(acc_chunk))
    )
    
    # Collect into RAM
    df_chunk = lazy_txns.collect()
    print(f" -> Loaded {df_chunk.shape[0]:,} rows for this chunk.")
    
    # 🚨 THE FIX: Convert text strings to actual Datetime objects! 🚨
    # strict=False ensures that if there's a corrupted timestamp, it turns to Null instead of crashing
    df_chunk = df_chunk.with_columns([
        pl.col("transaction_timestamp").str.to_datetime(strict=False)
    ])
    
    # Sort chronologically per account
    df_chunk = df_chunk.sort(["account_id", "transaction_timestamp"])
    
    # Calculate Time Gaps & Dates (The 'Micro-Ping' Engine)
    df_chunk = df_chunk.with_columns([
        pl.col("transaction_timestamp").diff().dt.total_seconds().over("account_id").alias("gap_sec"),
        pl.col("transaction_timestamp").dt.date().alias("tx_date")
    ])
    
    # Aggregate PDF Features
    chunk_agg = df_chunk.group_by("account_id").agg([
        pl.col("gap_sec").median().alias("median_tx_gap_sec"),
        pl.col("gap_sec").max().alias("max_tx_gap_sec"),
        (pl.col("gap_sec") < 60).sum().alias("micro_ping_count"),
        (pl.col("gap_sec") < (6 * 3600)).sum().alias("rapid_tx_count"),
        pl.len().alias("chunk_total_tx") # For ratio math later
    ])
    
    # Daily max burst
    daily_counts = df_chunk.group_by(["account_id", "tx_date"]).agg(pl.len().alias("daily_tx"))
    max_daily = daily_counts.group_by("account_id").agg(pl.col("daily_tx").max().alias("max_daily_tx"))
    
    # Merge the chunk aggregates
    chunk_final = chunk_agg.join(max_daily, on="account_id", how="left")
    temporal_aggregates.append(chunk_final)
    
    # STRICT GARBAGE COLLECTION
    del df_chunk, chunk_agg, daily_counts, max_daily
    gc.collect()

print("\n--- COMBINING ALL CHUNKS & MERGING TO MASTER ---")
final_temporal = pl.concat(temporal_aggregates)

master_df = master_df.join(final_temporal, on="account_id", how="left")

print("Applying PDF Composite Mathematical Ratios...")
master_df = master_df.with_columns([
    (pl.col("micro_ping_count") / (pl.col("chunk_total_tx") + 1)).fill_null(0).alias("micro_ping_ratio"),
    (pl.col("rapid_tx_count") / (pl.col("chunk_total_tx") + 1)).fill_null(0).alias("liquidation_intensity"),
    (pl.col("max_daily_tx") / (pl.col("chunk_total_tx") + 1)).fill_null(0).alias("burst_ratio"),
    (pl.col("median_tx_gap_sec") / 3600).fill_null(0).alias("median_tx_gap_hours")
])

# Save Final V4 Checkpoint
master_df.write_parquet("/kaggle/working/master_features_v4.parquet")

print(f"✅ V4 MASTER MATRIX SAVED! Shape: {master_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np

print("--- LOADING V4 MASTER MATRIX ---")
df_master = pl.read_parquet("/kaggle/working/master_features_v4.parquet").to_pandas()

# 1. Separate Train and Test
df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

print(f"Train Shape: {df_train.shape}")
print(f"Test Shape: {df_test.shape}")

# 2. Define Features and Clean Data
# STRICT LEAKAGE & PII DROP LIST
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address' # <--- No PII allowed!
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()

# Fill missing values
df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

num_features = [c for c in features if c not in cat_features]
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

X = df_train[features]
y = df_train['is_mule'].astype(int)

# 3. Validation Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 4. Train the Model
print("\n--- TRAINING V2 CATBOOST MODEL ---")
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    auto_class_weights='Balanced', 
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

# 5. Evaluate
val_preds = model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, val_preds)
print(f"\n✅ V2 VALIDATION ROC-AUC: {auc_score:.5f}")

# 6. Top 15 Feature Importances
importances = model.get_feature_importance()
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print("\n--- TOP 15 STRONGEST FEATURES ---")
print(feat_imp.head(15))

# 7. Generate Submission V3
print("\n--- GENERATING V3 SUBMISSION ---")
test_probs = model.predict_proba(df_test[features])[:, 1]

submission = pd.DataFrame({
    'account_id': df_test['account_id'],
    'is_mule': test_probs,
    'suspicious_start': '', 
    'suspicious_end': ''
})

output_csv = "/kaggle/working/DONE_WITHIT_submission_v3.csv"
submission.to_csv(output_csv, index=False)
print(f"✅ SUBMISSION SAVED: {output_csv}")

In [ ]:
import polars as pl
import pandas as pd

print("--- IGNITING LEVEL 6: THE TEMPORAL IoU HACK ---")

# 1. Load your Rank 10 Submission
sub_df = pd.read_csv("/kaggle/working/DONE_WITHIT_submission_v3.csv")

# 2. Identify the "Hard Mules" (Threshold: 0.85 to be highly precise)
# We only want to flag time windows for the accounts we are SURE are mules.
threshold = 0.85
mule_accounts = sub_df[sub_df['is_mule'] > threshold]['account_id'].tolist()
print(f"Identified {len(mule_accounts)} high-confidence mule accounts for timestamp extraction.")

# 3. Extract exact start and end times for JUST these accounts
TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"

print("Scanning transactions for mule timestamps...")
lazy_txns = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
)

# Convert strings to datetime
lazy_txns = lazy_txns.with_columns(
    pl.col("transaction_timestamp").str.to_datetime(strict=False)
)

# Get absolute first and last transaction for each mule account
mule_times = lazy_txns.group_by("account_id").agg([
    pl.col("transaction_timestamp").min().alias("start"),
    pl.col("transaction_timestamp").max().alias("end")
]).collect()

# Format strictly to YYYY-MM-DDTHH:MM:SS
mule_times = mule_times.with_columns([
    pl.col("start").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
])

# 4. Merge back into the submission
time_df = mule_times.to_pandas()

# Drop the old blank columns and merge the new ones
sub_df = sub_df.drop(columns=['suspicious_start', 'suspicious_end']) 
sub_df = sub_df.merge(time_df[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left')

# Fill NaNs with empty strings for normal accounts (required by guidelines)
sub_df['suspicious_start'] = sub_df['suspicious_start'].fillna('')
sub_df['suspicious_end'] = sub_df['suspicious_end'].fillna('')

# 5. Save Final V4 Submission
output_csv = "/kaggle/working/DONE_WITHIT_submission_v4_IoU.csv"
sub_df.to_csv(output_csv, index=False)

print(f"\n✅ TEMPORAL SUBMISSION SAVED: {output_csv}")
print("\nPreview of flagged mules with their predicted timestamps:")
print(sub_df[sub_df['is_mule'] > threshold].head())

In [ ]:
import polars as pl
import numpy as np
import gc

print("--- IGNITING LEVEL 7: NUMERICAL ALCHEMY (V2) ---")

# 1. Load V4 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v4.parquet")

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
TXN_ADD_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions_additional"

batch_stats = []

# 2. Map-Reduce for Balance Volatility & Concentration
for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    print(f"Analyzing patterns in {batch}...")
    
    t = pl.scan_parquet(f"{TXN_DIR}/{batch}/*.parquet").select(["transaction_id", "account_id", "amount", "counterparty_id"])
    ta = pl.scan_parquet(f"{TXN_ADD_DIR}/{batch}/*.parquet").select(["transaction_id", "balance_after_transaction"])
    
    # Joining to get the full picture
    joined = t.join(ta, on="transaction_id", how="inner")
    
    # Calculate Features
    batch_plan = (
        joined
        .group_by("account_id")
        .agg([
            # Balance Behavior (Step 3)
            pl.col("balance_after_transaction").std().alias("balance_std"),
            pl.col("balance_after_transaction").diff().abs().mean().alias("balance_avg_diff"),
            (pl.col("balance_after_transaction") < 100).sum().alias("near_zero_count"),
            
            # Counterparty Concentration (Step 1) - Max hits on a single peer
            pl.col("counterparty_id").value_counts().struct.field("count").max().alias("top_cpty_hits"),
            
            # Amount Stats (Step 4)
            pl.col("amount").std().alias("amount_std"),
            (pl.col("amount") % 1000 == 0).sum().alias("large_round_count")
        ])
    )
    
    batch_stats.append(batch_plan.collect(engine="streaming"))
    gc.collect()

# 3. Reduce & Feature Crosses
print("\n--- FINALIZING RATIOS & CROSS-FEATURES ---")
final_stats = (
    pl.concat(batch_stats)
    .group_by("account_id")
    .agg([
        pl.col("balance_std").mean().alias("balance_volatility"),
        pl.col("balance_avg_diff").mean().alias("balance_mobility"),
        pl.col("near_zero_count").sum().alias("liquidity_wipeout_freq"),
        pl.col("top_cpty_hits").max().alias("max_cpty_concentration"),
        pl.col("amount_std").mean().alias("amt_dispersion"),
        pl.col("large_round_count").sum().alias("heavy_structuring_count")
    ])
)

master_df = master_df.join(final_stats, on="account_id", how="left")

# 4. Level 7 Logic Injection (Composite Ratios)
master_df = master_df.with_columns([
    # Step 1: Concentration Ratio
    (pl.col("max_cpty_concentration") / (pl.col("total_tx_count") + 1)).alias("cpty_concentration_ratio"),
    
    # Step 2: Burst + Network Combo (The multiplier effect)
    (pl.col("burst_ratio") * pl.col("unique_counterparties")).alias("burst_network_intensity"),
    
    # Step 4: Amount Coefficient of Variation
    (pl.col("amt_dispersion") / (pl.col("total_txn_volume") + 1)).alias("amt_cv")
])

# Save V5
master_df.write_parquet("/kaggle/working/master_features_v5.parquet")
print(f"✅ V5 MASTER MATRIX SAVED! Shape: {master_df.shape}")

In [ ]:
import polars as pl
import gc

print("--- IGNITING LEVEL 8: PEER DEVIATION & STATIC-CROSS (V2) ---")

# 1. Load V5 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v5.parquet")

# 2. Fix the Nomination Flag and Product Count
# Mapping Y/N to 1/0 so we can actually use them in math
master_df = master_df.with_columns([
    pl.col("nomination_flag").map_elements(lambda x: 1 if x == "Y" else 0, return_dtype=pl.Int32).alias("nomination_val"),
    (pl.col("loan_count").fill_null(0) + pl.col("cc_count").fill_null(0) + pl.col("od_count").fill_null(0)).alias("total_products_count")
])

# 3. Calculate Branch-Level Peer Norms
branch_norms = (
    master_df
    .group_by("branch_code")
    .agg([
        pl.col("total_tx_count").mean().alias("branch_avg_tx"),
        pl.col("total_txn_volume").mean().alias("branch_avg_volume")
    ])
)

# 4. Apply Cross-Ratios and Deviations
master_df = (
    master_df
    .join(branch_norms, on="branch_code", how="left")
    .with_columns([
        # Product Ratio: Transactions vs Banking Products (Skeleton Score)
        (pl.col("total_tx_count") / (pl.col("total_products_count") + 1)).alias("tx_per_product_ratio"),
        
        # Peer Deviation
        (pl.col("total_tx_count") / (pl.col("branch_avg_tx") + 1)).alias("tx_branch_deviation"),
        (pl.col("total_txn_volume") / (pl.col("branch_avg_volume") + 1)).alias("vol_branch_deviation"),
        
        # Transaction to Balance Efficiency
        (pl.col("total_txn_volume") / (pl.col("avg_balance") + 1)).alias("volume_velocity_ratio"),
        
        # Static Risk Score: Engagement level
        (pl.col("total_products_count") * pl.col("nomination_val")).alias("engagement_score")
    ])
)

# Clean up
master_df = master_df.drop(["branch_avg_tx", "branch_avg_volume", "nomination_val"])

# SAVE V6 CHECKPOINT
master_df.write_parquet("/kaggle/working/master_features_v6.parquet")

print(f"✅ V6 MASTER MATRIX SAVED! Shape: {master_df.shape}")
print("\nFinal Engineered Column Sample:")
print(master_df.select(pl.col("^.*_ratio$|^.*_deviation$")).columns)

In [ ]:
import polars as pl
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
import gc

print("--- TRAINING THE V6 CHAMPION MODEL ---")
df_master = pl.read_parquet("/kaggle/working/master_features_v6.parquet").to_pandas()

# 1. Split Train/Test
df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

# 2. Final Drop List (Leakage + PII + Redundant IDs)
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()

# Fill missing
df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)
df_train[features] = df_train[features].fillna(-1)
df_test[features] = df_test[features].fillna(-1)

X = df_train[features]
y = df_train['is_mule'].astype(int)

# 3. Stratified Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

# 4. Train with Early Stopping
model = CatBoostClassifier(
    iterations=1500, # Increased for more features
    learning_rate=0.03,
    depth=7,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    cat_features=cat_features,
    early_stopping_rounds=100,
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

# 5. Evaluate
val_probs = model.predict_proba(X_val)[:, 1]
print(f"\n✅ V6 VALIDATION ROC-AUC: {roc_auc_score(y_val, val_probs):.5f}")

# 6. Feature Importance Check
importances = model.get_feature_importance()
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print("\n--- TOP 15 FEATURES (THE V6 IMPACT) ---")
print(feat_imp.head(15))

# 7. Generate Submission V5
test_probs = model.predict_proba(df_test[features])[:, 1]
submission = pd.DataFrame({
    'account_id': df_test['account_id'],
    'is_mule': test_probs,
    'suspicious_start': '', 
    'suspicious_end': ''
})
submission.to_csv("/kaggle/working/V6_PRE_GRAPH_SUB.csv", index=False)

In [ ]:
import pandas as pd
import polars as pl

print("--- FIXING COLUMNS & REFINING TEMPORAL WINDOWS ---")

# 1. Load the probabilities
sub_df = pd.read_csv("/kaggle/working/V6_PRE_GRAPH_SUB.csv")

# 2. Extract timestamps for high-confidence accounts (>0.80)
mule_accounts = sub_df[sub_df['is_mule'] > 0.80]['account_id'].tolist()

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
mule_times = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_raw"),
        pl.col("transaction_timestamp").max().alias("end_raw")
    ])
    .collect()
)

# 3. Format strictly & Drop raw helper columns
mule_times = mule_times.with_columns([
    pl.col("start_raw").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end_raw").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
]).select(["account_id", "suspicious_start", "suspicious_end"]).to_pandas()

# 4. Merge and ensure ONLY allowed columns are present
sub_df = sub_df.drop(columns=['suspicious_start', 'suspicious_end'])
final_sub = sub_df.merge(mule_times, on='account_id', how='left').fillna('')

# STRICT COLUMN ORDER
final_sub = final_sub[['account_id', 'is_mule', 'suspicious_start', 'suspicious_end']]

# 5. Save the Clean V6
output_path = "/kaggle/working/DONE_WITHIT_V6_CLEAN.csv"
final_sub.to_csv(output_path, index=False)

print(f"✅ CLEAN V6 SUBMISSION SAVED: {output_path}")
print(f"Columns in file: {final_sub.columns.tolist()}")

In [ ]:
import polars as pl
import gc

print("--- IGNITING LEVEL 9: GRAPH TOPOLOGY ENGINE ---")

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"

# 1. Build a Global Edge List (Who sends to Whom)
# We aggregate counts to reduce row count for the graph engine
print("Building global edge list...")
edges = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "counterparty_id"])
    .group_by(["account_id", "counterparty_id"])
    .len() # Number of times they interacted
    .collect(engine="streaming")
)

# 2. Calculate PageRank (Simplified Iterative Implementation)
# Accounts that receive money from many 'important' accounts get higher scores.
print("Calculating Global Importance (PageRank simulation)...")

# In-degree: How many people send money TO this account?
in_degree = edges.group_by("counterparty_id").agg(pl.col("len").sum().alias("global_in_degree"))
in_degree = in_degree.rename({"counterparty_id": "account_id"})

# Out-degree: How many people does this account send money TO?
out_degree = edges.group_by("account_id").agg(pl.col("len").sum().alias("global_out_degree"))

# 3. Network "Hub" Score
# Mules often act as 'Authorities' (many ins) or 'Hubs' (many outs)
master_df = pl.read_parquet("/kaggle/working/master_features_v6.parquet")

master_df = (
    master_df
    .join(in_degree, on="account_id", how="left")
    .join(out_degree, on="account_id", how="left")
    .with_columns([
        pl.col("global_in_degree").fill_null(0),
        pl.col("global_out_degree").fill_null(0)
    ])
    .with_columns([
        # The 'Mule Flow' Ratio
        (pl.col("global_in_degree") / (pl.col("global_out_degree") + 1)).alias("network_flow_ratio"),
        # Connectivity density
        (pl.col("global_in_degree") + pl.col("global_out_degree")).alias("total_network_connectivity")
    ])
)

# 4. Save V7 Checkpoint
master_df.write_parquet("/kaggle/working/master_features_v7.parquet")
print(f"✅ V7 GRAPH MATRIX SAVED! Shape: {master_df.shape}")

del edges, in_degree, out_degree
gc.collect()

In [ ]:
import polars as pl
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
import gc

print("--- TRAINING THE V4 PODIUM FINISHER (V7 MATRIX) ---")
df_master = pl.read_parquet("/kaggle/working/master_features_v7.parquet").to_pandas()

# 1. Final Split
df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

# 2. Drop List (PII + Leakage + Identifiers)
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region' # Dropping these to prevent regional overfitting
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()

# 3. Clean & Fill
df_train[features] = df_train[features].fillna(-1)
df_test[features] = df_test[features].fillna(-1)
df_train[cat_features] = df_train[cat_features].astype(str)
df_test[cat_features] = df_test[cat_features].astype(str)

X = df_train[features]
y = df_train['is_mule'].astype(int)

# 4. Stratified Split (80/20 for a robust validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 5. The Final Model Config
model = CatBoostClassifier(
    iterations=2000, 
    learning_rate=0.02,
    depth=8, # Deeper trees for complex graph interactions
    eval_metric='AUC',
    auto_class_weights='Balanced',
    cat_features=cat_features,
    early_stopping_rounds=150,
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

# 6. Evaluation
val_probs = model.predict_proba(X_val)[:, 1]
print(f"\n✅ V7 VALIDATION ROC-AUC: {roc_auc_score(y_val, val_probs):.6f}")

# 7. Final Rank Check - Feature Importances
importances = model.get_feature_importance()
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print("\n--- THE TOP 10 MULE DETECTORS (V7) ---")
print(feat_imp.head(10))

# 8. PREPARE THE RAW PREDICTIONS
test_probs = model.predict_proba(df_test[features])[:, 1]
submission = pd.DataFrame({
    'account_id': df_test['account_id'],
    'is_mule': test_probs,
    'suspicious_start': '', 
    'suspicious_end': ''
})
submission.to_csv("/kaggle/working/V7_RAW_PROBS.csv", index=False)

In [ ]:
import pandas as pd
import polars as pl

print("--- EXECUTING FINAL TEMPORAL ALIGNMENT ---")

# 1. Load V7 Raw Probs
sub_df = pd.read_csv("/kaggle/working/V7_RAW_PROBS.csv")

# 2. Extract high-confidence accounts for IoU scoring
# Using 0.75 to widen the net for Temporal IoU points
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()
print(f"Applying temporal windows to {len(mule_accounts)} suspicious accounts.")

# 3. Fetch exact timestamps from the source
TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
mule_times = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ])
    .collect()
)

# 4. Format to ISO-8601 strings
mule_times = mule_times.with_columns([
    pl.col("start_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
]).select(["account_id", "suspicious_start", "suspicious_end"]).to_pandas()

# 5. Final Merge & Column Clean-up
sub_df = sub_df.drop(columns=['suspicious_start', 'suspicious_end'])
final_sub = sub_df.merge(mule_times, on='account_id', how='left').fillna('')
final_sub = final_sub[['account_id', 'is_mule', 'suspicious_start', 'suspicious_end']]

# 6. SAVE THE PODIUM FILE
final_sub.to_csv("/kaggle/working/DONE_WITHIT_FINAL_PODIUM.csv", index=False)
print("✅ FINAL PODIUM SUBMISSION SAVED!")

In [ ]:
import pandas as pd

# 1. Load your two best models
v6_sub = pd.read_csv("/kaggle/working/DONE_WITHIT_V6_CLEAN.csv")
v7_sub = pd.read_csv("/kaggle/working/DONE_WITHIT_FINAL_PODIUM.csv")

# 2. Weighted Blend (60% V7 for precision, 40% V6 for raw AUC)
v6_sub['is_mule'] = (v7_sub['is_mule'] * 0.6) + (v6_sub['is_mule'] * 0.4)

# 3. Final Polish
# We keep the V7 temporal windows because they are more refined
v6_sub['suspicious_start'] = v7_sub['suspicious_start']
v6_sub['suspicious_end'] = v7_sub['suspicious_end']

v6_sub.to_csv("/kaggle/working/DONE_WITHIT_SUPER_ENSEMBLE.csv", index=False)
print("✅ SUPER ENSEMBLE GENERATED!")

In [ ]:
import polars as pl
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- IGNITING THE XGBOOST ENGINE (V2) ---")
df_master = pl.read_parquet("/kaggle/working/master_features_v7.parquet").to_pandas()

# 1. Split Train/Test
df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

# 2. Drop List
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

# 3. STRICT TYPE ENCODING (The Fix)
# Numerics get the -1
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

# Categoricals get a strictly typed string before converting to Pandas categories
df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

for c in cat_features:
    df_train[c] = df_train[c].astype('category')
    df_test[c] = df_test[c].astype('category')

X = df_train[features]
y = df_train['is_mule'].astype(int)

# 4. Stratified Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 5. Train XGBoost
print("Training XGBoost Classifier...")
xgb_model = xgb.XGBClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=7,
    scale_pos_weight=15, # Approximating 'Balanced' for highly imbalanced fraud
    tree_method='hist',
    enable_categorical=True,
    early_stopping_rounds=100,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

# 6. Evaluate XGBoost
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]
print(f"\n✅ XGBOOST VALIDATION ROC-AUC: {roc_auc_score(y_val, xgb_val_probs):.6f}")

# 7. Generate XGBoost Test Probs
xgb_test_probs = xgb_model.predict_proba(df_test[features])[:, 1]

# 8. THE ULTIMATE BLEND (CatBoost + XGBoost)
print("\n--- BLENDING INTO THE MASTER ENSEMBLE ---")
# Load your current best Super Ensemble (which already has V6 + V7 CatBoost)
current_best_sub = pd.read_csv("/kaggle/working/DONE_WITHIT_SUPER_ENSEMBLE.csv")

# Blend: 60% Current Best CatBoost Ensemble, 40% New XGBoost
final_probs = (current_best_sub['is_mule'] * 0.6) + (xgb_test_probs * 0.4)

# Build the final file
final_submission = pd.DataFrame({
    'account_id': current_best_sub['account_id'],
    'is_mule': final_probs,
    'suspicious_start': current_best_sub['suspicious_start'], # Keep our elite temporal windows
    'suspicious_end': current_best_sub['suspicious_end']
})

output_path = "/kaggle/working/DONE_WITHIT_XGB_BLEND.csv"
final_submission.to_csv(output_path, index=False)
print(f"✅ XGBOOST BLEND SAVED: {output_path}")

In [ ]:
import polars as pl
import gc

print("--- IGNITING LEVEL 10: WASH DYNAMICS & BENFORD'S LAW (V2) ---")

# 1. Load V7 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v7.parquet")

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
batch_stats = []

for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    print(f"Extracting Cartel Signatures in {batch}...")
    
    # 2. Scan transactions (IP address removed from scan!)
    lazy_txns = pl.scan_parquet(f"{TXN_DIR}/{batch}/*.parquet").select([
        "account_id", "amount", "txn_type"
    ])
    
    batch_plan = (
        lazy_txns
        .with_columns([
            # Extract first digit for Benford's Law
            pl.col("amount").cast(pl.String).str.slice(0, 1).cast(pl.Int32, strict=False).alias("first_digit"),
            
            # Split Credit and Debit amounts
            pl.when(pl.col("txn_type").cast(pl.String).str.contains("C")).then(pl.col("amount")).otherwise(0).alias("credit_amt"),
            pl.when(pl.col("txn_type").cast(pl.String).str.contains("D")).then(pl.col("amount")).otherwise(0).alias("debit_amt")
        ])
        .group_by("account_id")
        .agg([
            pl.col("credit_amt").sum().alias("total_credit"),
            pl.col("debit_amt").sum().alias("total_debit"),
            
            # Benford Proxy: Count how many times the first digit is a 1 or 2
            (pl.col("first_digit").is_in([1, 2])).sum().alias("benford_1_2_count")
        ])
    )
    
    batch_stats.append(batch_plan.collect(engine="streaming"))
    gc.collect()

print("\n--- FINALIZING LEVEL 10 MATH ---")
# 3. Reduce all batches
final_stats = (
    pl.concat(batch_stats)
    .group_by("account_id")
    .agg([
        pl.col("total_credit").sum().alias("global_credit"),
        pl.col("total_debit").sum().alias("global_debit"),
        pl.col("benford_1_2_count").sum().alias("total_benford_matches")
    ])
)

# 4. Inject the advanced ratios into Master
master_df = master_df.join(final_stats, on="account_id", how="left")

master_df = master_df.with_columns([
    # The Wash Ratio: Approaching 1.0 means money goes out exactly as fast as it comes in
    (pl.col("global_debit") / (pl.col("global_credit") + 1)).alias("wash_pass_through_ratio"),
    
    # Benford Deviation: Ratio of "natural" starting numbers vs total txns
    (pl.col("total_benford_matches") / (pl.col("total_tx_count") + 1)).alias("benford_compliance_score"),
    
    # VPN Hopping Score (Using the 'unique_ips' column we ALREADY have in master_df!)
    (pl.col("unique_ips").fill_null(0) / (pl.col("total_tx_count") + 1)).alias("ip_entropy")
])

# 5. Save V8 Checkpoint
master_df.write_parquet("/kaggle/working/master_features_v8.parquet")
print(f"✅ V8 MASTER MATRIX SAVED! Shape: {master_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- 🏆 THE FINAL ENDGAME: V8 TRI-FORCE ENSEMBLE ---")

# 1. Load the Ultimate V8 Matrix
df_master = pl.read_parquet("/kaggle/working/master_features_v8.parquet").to_pandas()

df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

# Clean & Type (Strictly enforcing for both XGBoost and CatBoost compatibility)
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

for c in cat_features:
    df_train[c] = df_train[c].astype('category')
    df_test[c] = df_test[c].astype('category')

X = df_train[features]
y = df_train['is_mule'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 2. Train CatBoost
print("\n[1/4] Training CatBoost Champion...")
cat_model = CatBoostClassifier(
    iterations=2000, learning_rate=0.02, depth=8,
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=150, verbose=200
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
cat_val_probs = cat_model.predict_proba(X_val)[:, 1]
print(f"✅ CatBoost Val AUC: {roc_auc_score(y_val, cat_val_probs):.6f}")

# 3. Train XGBoost
print("\n[2/4] Training XGBoost Heavyweight...")
xgb_model = xgb.XGBClassifier(
    n_estimators=1500, learning_rate=0.03, max_depth=7,
    scale_pos_weight=15, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=100, eval_metric='auc', random_state=42
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]
print(f"✅ XGBoost Val AUC: {roc_auc_score(y_val, xgb_val_probs):.6f}")

# 4. Blend Predictions
print("\n[3/4] Blending Predictions (60% CatBoost / 40% XGBoost)...")
cat_test_probs = cat_model.predict_proba(df_test[features])[:, 1]
xgb_test_probs = xgb_model.predict_proba(df_test[features])[:, 1]

final_probs = (cat_test_probs * 0.6) + (xgb_test_probs * 0.4)

sub_df = pd.DataFrame({'account_id': df_test['account_id'], 'is_mule': final_probs})

# 5. Extract Temporal Windows
print("\n[4/4] Aligning Temporal Windows...")
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
mule_times = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ])
    .collect()
)

mule_times = mule_times.with_columns([
    pl.col("start_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
]).select(["account_id", "suspicious_start", "suspicious_end"]).to_pandas()

final_sub = sub_df.merge(mule_times, on='account_id', how='left').fillna('')
final_sub = final_sub[['account_id', 'is_mule', 'suspicious_start', 'suspicious_end']]

output_path = "/kaggle/working/DONE_WITHIT_V8_ULTIMATE.csv"
final_sub.to_csv(output_path, index=False)
print(f"\n🏆 ULTIMATE V8 SUBMISSION READY: {output_path}")

In [ ]:
import polars as pl
import gc

print("--- IGNITING LEVEL 11: THE BEHAVIORAL FINGERPRINT ENGINE ---")

# 1. Load V8 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v8.parquet")
TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"

batch_stats = []

for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    print(f"Extracting Behavioral Fingerprints in {batch}...")
    
    # 2. Scan transactions and parse datetime
    lazy_txns = (
        pl.scan_parquet(f"{TXN_DIR}/{batch}/*.parquet")
        .select(["account_id", "transaction_timestamp", "amount", "txn_type"])
        .with_columns([
            pl.col("transaction_timestamp").str.to_datetime(strict=False).alias("ts")
        ])
        .with_columns([
            pl.col("ts").dt.hour().alias("hour"),
            pl.when(pl.col("txn_type").cast(pl.String).str.contains("C")).then(pl.col("amount")).otherwise(0).alias("credit_amt"),
            pl.when(pl.col("txn_type").cast(pl.String).str.contains("D")).then(pl.col("amount")).otherwise(0).alias("debit_amt")
        ])
        .with_columns([
            # Night Owl Flag (Transactions between Midnight and 5 AM)
            ((pl.col("hour") >= 0) & (pl.col("hour") <= 5)).cast(pl.Int32).alias("is_night_txn")
        ])
        .group_by("account_id")
        .agg([
            # Temporal Routine Fingerprint
            pl.col("hour").std().alias("hour_std_dev"),
            pl.col("is_night_txn").sum().alias("night_txn_count"),
            
            # Amount Extremes (Catching the massive one-off cartel dumps)
            pl.col("credit_amt").max().alias("max_single_credit"),
            pl.col("debit_amt").max().alias("max_single_debit")
        ])
    )
    
    batch_stats.append(lazy_txns.collect(engine="streaming"))
    gc.collect()

print("\n--- REDUCING & INJECTING LEVEL 11 RATIOS ---")

# 3. Reduce across all batches
final_stats = (
    pl.concat(batch_stats)
    .group_by("account_id")
    .agg([
        pl.col("hour_std_dev").mean().alias("avg_hour_variance"),
        pl.col("night_txn_count").sum().alias("total_night_txns"),
        pl.col("max_single_credit").max().alias("global_max_credit"),
        pl.col("max_single_debit").max().alias("global_max_debit")
    ])
)

master_df = master_df.join(final_stats, on="account_id", how="left")

# 4. The Final Master Ratios
master_df = master_df.with_columns([
    # 1. The Vampire Ratio: What % of their activity happens in the dead of night?
    (pl.col("total_night_txns") / (pl.col("total_tx_count") + 1)).alias("vampire_ratio"),
    
    # 2. The Exact Flush Ratio: Does their biggest debit instantly wipe out their biggest credit?
    (pl.col("global_max_debit") / (pl.col("global_max_credit") + 1)).alias("max_flush_ratio")
])

# 5. Save V9 Checkpoint
master_df.write_parquet("/kaggle/working/master_features_v9.parquet")
print(f"✅ V9 MASTER MATRIX SAVED! Shape: {master_df.shape}")

In [ ]:
import polars as pl

print("--- 🧠 INJECTING DEGREE Z-SCORE (THE ANOMALY HUNTER) ---")

# 1. Load V9
master_df = pl.read_parquet("/kaggle/working/master_features_v9.parquet")

# 2. Calculate Branch-level Network Degree Stats
branch_degree_stats = master_df.group_by("branch_code").agg([
    pl.col("unique_counterparties").mean().alias("branch_avg_degree"),
    pl.col("unique_counterparties").std().alias("branch_std_degree")
])

# 3. Merge and Calculate Z-Score
master_df = master_df.join(branch_degree_stats, on="branch_code", how="left")

master_df = master_df.with_columns([
    # Z-Score Formula applied to network connections
    # Adding a tiny epsilon (0.0001) to prevent division by zero in branches where everyone has the exact same degree
    ((pl.col("unique_counterparties") - pl.col("branch_avg_degree")) / 
     (pl.col("branch_std_degree") + 0.0001)).alias("degree_zscore")
]).drop(["branch_avg_degree", "branch_std_degree"]) # Clean up temp columns

# 4. Save V10
master_df.write_parquet("/kaggle/working/master_features_v10.parquet")
print(f"✅ V10 MATRIX SAVED WITH DEGREE Z-SCORE! Shape: {master_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- 🚀 IGNITING THE V10 TRI-FORCE ENSEMBLE ---")

# 1. Load the V10 Matrix
df_master = pl.read_parquet("/kaggle/working/master_features_v10.parquet").to_pandas()

df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

# Clean & Type 
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

for c in cat_features:
    df_train[c] = df_train[c].astype('category')
    df_test[c] = df_test[c].astype('category')

X = df_train[features]
y = df_train['is_mule'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 2. Train CatBoost 
print("\n[1/4] Training CatBoost Champion...")
cat_model = CatBoostClassifier(
    iterations=2000, learning_rate=0.02, depth=8,
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=150, verbose=200
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
cat_val_probs = cat_model.predict_proba(X_val)[:, 1]
print(f"✅ CatBoost Val AUC: {roc_auc_score(y_val, cat_val_probs):.6f}")

# 3. Train XGBoost 
print("\n[2/4] Training XGBoost Heavyweight...")
xgb_model = xgb.XGBClassifier(
    n_estimators=1500, learning_rate=0.03, max_depth=7,
    scale_pos_weight=15, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=100, eval_metric='auc', random_state=42
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]
print(f"✅ XGBoost Val AUC: {roc_auc_score(y_val, xgb_val_probs):.6f}")

# 4. Blend Predictions
print("\n[3/4] Blending Predictions (60% CatBoost / 40% XGBoost)...")
cat_test_probs = cat_model.predict_proba(df_test[features])[:, 1]
xgb_test_probs = xgb_model.predict_proba(df_test[features])[:, 1]

final_probs = (cat_test_probs * 0.6) + (xgb_test_probs * 0.4)

sub_df = pd.DataFrame({'account_id': df_test['account_id'], 'is_mule': final_probs})

# 5. Extract Temporal Windows
print("\n[4/4] Aligning Temporal Windows...")
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
mule_times = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ])
    .collect()
)

mule_times = mule_times.with_columns([
    pl.col("start_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
]).select(["account_id", "suspicious_start", "suspicious_end"]).to_pandas()

final_sub = sub_df.merge(mule_times, on='account_id', how='left').fillna('')
final_sub = final_sub[['account_id', 'is_mule', 'suspicious_start', 'suspicious_end']]

output_path = "/kaggle/working/DONE_WITHIT_V10_PUBLIC.csv"
final_sub.to_csv(output_path, index=False)
print(f"\n🏆 V10 PUBLIC SUBMISSION READY: {output_path}")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import gc

print("--- 🧠 IGNITING LEVEL 13: BOT SIGNATURES & UNSUPERVISED ANOMALY ---")

# 1. Load V10 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v10.parquet")
TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"

batch_stats = []

for batch in ["batch-1", "batch-2", "batch-3", "batch-4"]:
    print(f"Extracting Bot Signatures in {batch}...")
    
    # 2. Extract Exact Seconds and Minutes for Bot Detection
    lazy_txns = (
        pl.scan_parquet(f"{TXN_DIR}/{batch}/*.parquet")
        .select(["account_id", "transaction_timestamp"])
        .with_columns([
            pl.col("transaction_timestamp").str.to_datetime(strict=False).alias("ts")
        ])
        .with_columns([
            pl.col("ts").dt.minute().alias("minute"),
            pl.col("ts").dt.second().alias("second")
        ])
        .group_by("account_id")
        .agg([
            pl.col("minute").n_unique().alias("unique_minutes"),
            pl.col("second").n_unique().alias("unique_seconds"),
            pl.col("second").std().alias("second_variance")
        ])
    )
    
    batch_stats.append(lazy_txns.collect(engine="streaming"))
    gc.collect()

print("\n--- INJECTING BOT RATIOS ---")
# 3. Reduce Time-Series Stats
final_stats = (
    pl.concat(batch_stats)
    .group_by("account_id")
    .agg([
        pl.col("unique_minutes").sum().alias("total_unique_minutes"),
        pl.col("unique_seconds").sum().alias("total_unique_seconds"),
        pl.col("second_variance").mean().alias("avg_second_variance")
    ])
)

master_df = master_df.join(final_stats, on="account_id", how="left")

# Bot Ratios: Low uniqueness per transaction = Automated Script
master_df = master_df.with_columns([
    (pl.col("total_unique_seconds") / (pl.col("total_tx_count") + 1)).alias("second_entropy_ratio"),
    (pl.col("total_unique_minutes") / (pl.col("total_tx_count") + 1)).alias("minute_entropy_ratio"),
    
    # The pure 'Mule Score' from your script!
    (pl.col("wash_pass_through_ratio") * pl.col("burst_ratio")).alias("relay_amplification_score")
])

print("\n--- EXECUTING UNSUPERVISED ISOLATION FOREST ---")
# 4. The Isolation Forest Meta-Feature Injection
# Convert to Pandas temporarily for sklearn
df_pandas = master_df.to_pandas()

# Select numeric features for Anomaly Detection (ignoring IDs and target)
num_cols = df_pandas.select_dtypes(include=[np.number]).columns.tolist()
if 'is_mule' in num_cols: num_cols.remove('is_mule')

# Fill NaNs for Sklearn
X_iso = df_pandas[num_cols].fillna(-1)

# Train Isolation Forest (just like in your pipeline.py!)
iso = IsolationForest(n_estimators=150, contamination=0.02, random_state=42, n_jobs=-1)
iso.fit(X_iso)

# Generate anomaly scores (lower is more anomalous, we negate it so higher = riskier)
df_pandas["iso_anomaly_score"] = -iso.score_samples(X_iso)

# Convert back to Polars to save
master_df = pl.from_pandas(df_pandas)

# 5. Save V11 Checkpoint
master_df.write_parquet("/kaggle/working/master_features_v11.parquet")
print(f"✅ V11 MATRIX SAVED! Shape: {master_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- 🚀 IGNITING THE V11 TRI-FORCE ENSEMBLE ---")

# 1. Load the V11 Matrix
df_master = pl.read_parquet("/kaggle/working/master_features_v11.parquet").to_pandas()

df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

# Clean & Type 
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

for c in cat_features:
    df_train[c] = df_train[c].astype('category')
    df_test[c] = df_test[c].astype('category')

X = df_train[features]
y = df_train['is_mule'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 2. Train CatBoost 
print("\n[1/4] Training CatBoost Champion...")
cat_model = CatBoostClassifier(
    iterations=2000, learning_rate=0.02, depth=8,
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=150, verbose=200
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
cat_val_probs = cat_model.predict_proba(X_val)[:, 1]
print(f"✅ CatBoost Val AUC: {roc_auc_score(y_val, cat_val_probs):.6f}")

# 3. Train XGBoost 
print("\n[2/4] Training XGBoost Heavyweight...")
xgb_model = xgb.XGBClassifier(
    n_estimators=1500, learning_rate=0.03, max_depth=7,
    scale_pos_weight=15, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=100, eval_metric='auc', random_state=42
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]
print(f"✅ XGBoost Val AUC: {roc_auc_score(y_val, xgb_val_probs):.6f}")

# 4. Blend Predictions
print("\n[3/4] Blending Predictions (60% CatBoost / 40% XGBoost)...")
cat_test_probs = cat_model.predict_proba(df_test[features])[:, 1]
xgb_test_probs = xgb_model.predict_proba(df_test[features])[:, 1]

final_probs = (cat_test_probs * 0.6) + (xgb_test_probs * 0.4)

sub_df = pd.DataFrame({'account_id': df_test['account_id'], 'is_mule': final_probs})

# 5. Extract Temporal Windows
print("\n[4/4] Aligning Temporal Windows...")
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
mule_times = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ])
    .collect()
)

mule_times = mule_times.with_columns([
    pl.col("start_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
]).select(["account_id", "suspicious_start", "suspicious_end"]).to_pandas()

final_sub = sub_df.merge(mule_times, on='account_id', how='left').fillna('')
final_sub = final_sub[['account_id', 'is_mule', 'suspicious_start', 'suspicious_end']]

output_path = "/kaggle/working/DONE_WITHIT_V11_PUBLIC.csv"
final_sub.to_csv(output_path, index=False)
print(f"\n🏆 V11 PUBLIC SUBMISSION READY: {output_path}")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import gc

print("--- 🕵️‍♂️ IGNITING LEVEL 14: THE BURNER IDENTITY ENGINE (V4 - ULTRA BULLETPROOF) ---")

# 1. Load V11 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v11.parquet")
DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

print("Loading Static Tables via Pandas...")
accounts = pd.read_parquet(f"{DATA_PATH}/accounts.parquet")
customers = pd.read_parquet(f"{DATA_PATH}/customers.parquet")
linkage = pd.read_parquet(f"{DATA_PATH}/customer_account_linkage.parquet")
branch = pd.read_parquet(f"{DATA_PATH}/branch.parquet")
demographics = pd.read_parquet(f"{DATA_PATH}/demographics.parquet")
accounts_add = pd.read_parquet(f"{DATA_PATH}/accounts-additional.parquet")

# 2. Extract First Transaction Time in Polars
print("Extracting Account Timestamps via Polars...")
first_tx = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg(pl.col("transaction_timestamp").min().alias("first_tx_time"))
    .collect()
    .to_pandas()
)

print("Merging the Entire Data Warehouse...")
# Merge ALL columns so we don't get KeyErrors guessing the names
df_static = accounts.copy()
df_static = df_static.merge(accounts_add, on="account_id", how="left")
df_static = df_static.merge(linkage, on="account_id", how="left")
df_static = df_static.merge(customers, on="customer_id", how="left")
df_static = df_static.merge(branch, on="branch_code", how="left")
df_static = df_static.merge(demographics, on="customer_id", how="left")
df_static = df_static.merge(first_tx, on="account_id", how="left")

print("Dynamically Hunting for Burner Signatures...")

# --- DYNAMIC FEATURE ENGINEERING ---

# 1. KYC Strength
kyc_cols = ["pan_available", "aadhaar_available", "passport_available"]
df_static["kyc_strength"] = 0
for col in kyc_cols:
    if col in df_static.columns:
        df_static["kyc_strength"] += (df_static[col] == "Y").astype(int)

# 2. Digital Usage
digital_cols = ["mobile_banking_flag", "internet_banking_flag", "atm_card_flag"]
df_static["digital_usage"] = 0
for col in digital_cols:
    if col in df_static.columns:
        df_static["digital_usage"] += (df_static[col] == "Y").astype(int)

# 3. State PIN Mismatch (Hunting down the exact column names)
c_pin = "customer_pin" if "customer_pin" in df_static.columns else "permanent_pin"
b_pin = "branch_pin" if "branch_pin" in df_static.columns else "branch_pin_code"

if c_pin in df_static.columns and b_pin in df_static.columns:
    df_static["customer_pin_str"] = df_static[c_pin].astype(str).str[:2]
    df_static["branch_pin_str"] = df_static[b_pin].astype(str).str[:2]
    df_static["state_mismatch"] = (df_static["customer_pin_str"] != df_static["branch_pin_str"]).astype(int)
else:
    df_static["state_mismatch"] = 0 # Fallback

# 4. Mobile Update Spike
if "last_mobile_update_date" in df_static.columns:
    df_static["last_mobile_update_date"] = pd.to_datetime(df_static["last_mobile_update_date"], errors="coerce")
    df_static["mobile_update_spike_delay"] = (df_static["first_tx_time"] - df_static["last_mobile_update_date"]).dt.total_seconds() / 3600
    df_static["mobile_update_spike_flag"] = (df_static["mobile_update_spike_delay"] < 24).astype(int)
else:
    df_static["mobile_update_spike_delay"] = -1
    df_static["mobile_update_spike_flag"] = 0

# Extract only the final engineered features
final_static = df_static[[
    "account_id", "kyc_strength", "digital_usage", "state_mismatch", 
    "mobile_update_spike_delay", "mobile_update_spike_flag"
]]

print("Injecting into Master Matrix...")
final_static_pl = pl.from_pandas(final_static)
master_df = master_df.join(final_static_pl, on="account_id", how="left")

# Save V12 Checkpoint
master_df.write_parquet("/kaggle/working/master_features_v12.parquet")
print(f"✅ V12 MATRIX SAVED WITH STATIC BURNER PROFILES! Shape: {master_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- 🚀 IGNITING THE V12 TRI-FORCE ENSEMBLE (THE PODIUM PUSH) ---")

# 1. Load the V12 Matrix
df_master = pl.read_parquet("/kaggle/working/master_features_v12.parquet").to_pandas()

df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

# Clean & Type 
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

for c in cat_features:
    df_train[c] = df_train[c].astype('category')
    df_test[c] = df_test[c].astype('category')

X = df_train[features]
y = df_train['is_mule'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# 2. Train CatBoost 
print("\n[1/4] Training CatBoost Champion...")
cat_model = CatBoostClassifier(
    iterations=2000, learning_rate=0.02, depth=8,
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=150, verbose=200
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
cat_val_probs = cat_model.predict_proba(X_val)[:, 1]
print(f"✅ CatBoost Val AUC: {roc_auc_score(y_val, cat_val_probs):.6f}")

# 3. Train XGBoost 
print("\n[2/4] Training XGBoost Heavyweight...")
xgb_model = xgb.XGBClassifier(
    n_estimators=1500, learning_rate=0.03, max_depth=7,
    scale_pos_weight=15, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=100, eval_metric='auc', random_state=42
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]
print(f"✅ XGBoost Val AUC: {roc_auc_score(y_val, xgb_val_probs):.6f}")

# 4. Blend Predictions
print("\n[3/4] Blending Predictions (60% CatBoost / 40% XGBoost)...")
cat_test_probs = cat_model.predict_proba(df_test[features])[:, 1]
xgb_test_probs = xgb_model.predict_proba(df_test[features])[:, 1]

final_probs = (cat_test_probs * 0.6) + (xgb_test_probs * 0.4)

sub_df = pd.DataFrame({'account_id': df_test['account_id'], 'is_mule': final_probs})

# 5. Extract Temporal Windows
print("\n[4/4] Aligning Temporal Windows...")
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
mule_times = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ])
    .collect()
)

mule_times = mule_times.with_columns([
    pl.col("start_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
]).select(["account_id", "suspicious_start", "suspicious_end"]).to_pandas()

final_sub = sub_df.merge(mule_times, on='account_id', how='left').fillna('')
final_sub = final_sub[['account_id', 'is_mule', 'suspicious_start', 'suspicious_end']]

output_path = "/kaggle/working/DONE_WITHIT_V12_PUBLIC.csv"
final_sub.to_csv(output_path, index=False)
print(f"\n🏆 V12 PUBLIC SUBMISSION READY: {output_path}")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- ☢️ IGNITING LEVEL 15: THE PSEUDO-LABELING NUKE ---")

# 1. Load V12 Matrix and our best Public predictions
df_master = pl.read_parquet("/kaggle/working/master_features_v12.parquet").to_pandas()
v12_preds = pd.read_csv("/kaggle/working/DONE_WITHIT_V12_PUBLIC.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

print(f"Original Train Size: {len(df_train_orig)}")

# 2. EXTRACT PSEUDO-LABELS (The 99.9% Confidence Bounds)
# We only take the absolute extremes to avoid poisoning our model with bad guesses
confident_mules_ids = v12_preds[v12_preds['is_mule'] > 0.995]['account_id']
confident_legit_ids = v12_preds[v12_preds['is_mule'] < 0.005]['account_id']

print(f"Adding {len(confident_mules_ids)} confident Mules and {len(confident_legit_ids)} confident Legit accounts from Test Set...")

# Grab those rows from the test set
pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules_ids)].copy()
pseudo_mules['is_mule'] = 1.0

pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit_ids)].copy()
pseudo_legit['is_mule'] = 0.0

# 3. FORGE THE SUPER-DATASET
df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)
df_test = df_test_orig.copy() # We still need to predict on the full test set

print(f"New Super-Train Size: {len(df_train)}")

# 4. Standard Prep
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str)
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str)

for c in cat_features:
    df_train[c] = df_train[c].astype('category')
    df_test[c] = df_test[c].astype('category')

X = df_train[features]
y = df_train['is_mule'].astype(int)

# Use original train set for Validation to ensure we don't overfit to our own pseudo-labels
X_train_split = df_train_orig[features].fillna(-1)
y_train_split = df_train_orig['is_mule'].astype(int)

for c in cat_features:
    X_train_split[c] = X_train_split[c].astype(str).astype('category')

_, X_val, _, y_val = train_test_split(X_train_split, y_train_split, test_size=0.15, stratify=y_train_split, random_state=42)

# 5. Train CatBoost on Super-Dataset
print("\n[1/4] Training CatBoost Pseudo-Champion...")
cat_model = CatBoostClassifier(
    iterations=2500, learning_rate=0.015, depth=8, # Slightly lower LR, more iterations for the bigger dataset
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=150, verbose=250
)
cat_model.fit(X, y, eval_set=(X_val, y_val), use_best_model=True)

# 6. Train XGBoost on Super-Dataset
print("\n[2/4] Training XGBoost Pseudo-Heavyweight...")
xgb_model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.02, max_depth=7,
    scale_pos_weight=15, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=100, eval_metric='auc', random_state=42
)
xgb_model.fit(X, y, eval_set=[(X_val, y_val)], verbose=250)

# 7. Blend Predictions
print("\n[3/4] Blending Predictions (60% CatBoost / 40% XGBoost)...")
cat_test_probs = cat_model.predict_proba(df_test[features])[:, 1]
xgb_test_probs = xgb_model.predict_proba(df_test[features])[:, 1]

final_probs = (cat_test_probs * 0.6) + (xgb_test_probs * 0.4)
sub_df = pd.DataFrame({'account_id': df_test['account_id'], 'is_mule': final_probs})

# 8. Extract Temporal Windows
print("\n[4/4] Aligning Temporal Windows...")
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()

TXN_DIR = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2/transactions"
mule_times = (
    pl.scan_parquet(f"{TXN_DIR}/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ])
    .collect()
)

mule_times = mule_times.with_columns([
    pl.col("start_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_start"),
    pl.col("end_ts").dt.strftime("%Y-%m-%dT%H:%M:%S").alias("suspicious_end")
]).select(["account_id", "suspicious_start", "suspicious_end"]).to_pandas()

final_sub = sub_df.merge(mule_times, on='account_id', how='left').fillna('')
final_sub = final_sub[['account_id', 'is_mule', 'suspicious_start', 'suspicious_end']]

output_path = "/kaggle/working/DONE_WITHIT_V13_PSEUDO.csv"
final_sub.to_csv(output_path, index=False)
print(f"\n🏆 V13 PSEUDO-LABELED SUBMISSION READY: {output_path}")

In [ ]:
import polars as pl
import gc

print("--- 🕸️ IGNITING LEVEL 16: THE BOTNET COLLISION ENGINE (V2) ---")

# 1. Load V12 Checkpoint
master_df = pl.read_parquet("/kaggle/working/master_features_v12.parquet")
DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

# 2. Scan IPs and Accounts simultaneously
print("Mapping IP-to-Account Collisions across Batches...")
# We need to join the main transaction with transactions_additional to get IP + AccountID
# Note: Using part_*.parquet inside each batch folder
txn_scan = pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet").select(["transaction_id", "account_id"])
add_scan = pl.scan_parquet(f"{DATA_PATH}/transactions_additional/batch-*/*.parquet").select(["transaction_id", "ip_address"])

# 3. Build the Bipartite IP-Account Map
# We only care about unique pairs to find how many accounts use one IP
ip_account_map = (
    txn_scan.join(add_scan, on="transaction_id", how="inner")
    .select(["account_id", "ip_address"])
    .drop_nulls("ip_address")
    .unique()
)

# 4. Count Accounts per IP (Collisions)
ip_collisions = (
    ip_account_map.group_by("ip_address")
    .agg(pl.col("account_id").n_unique().alias("accounts_on_this_ip"))
)

# 5. Map the Max Collision back to the Account
print("Calculating Max IP Sharing Node...")
account_ip_risk = (
    ip_account_map.join(ip_collisions, on="ip_address", how="left")
    .group_by("account_id")
    .agg(pl.col("accounts_on_this_ip").max().alias("max_ip_sharing_node"))
    .collect(streaming=True)
)

# 6. Inject into Master and Save
master_df = master_df.join(account_ip_risk, on="account_id", how="left")
master_df = master_df.with_columns([
    (pl.col("max_ip_sharing_node").fill_null(1)).alias("max_ip_sharing_node")
])

master_df.write_parquet("/kaggle/working/master_features_v13.parquet")
print(f"✅ V13 MATRIX SAVED! Shape: {master_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- 🏆 THE FINAL GOLD MEDAL RETRAIN: V13 ---")

# 1. Load the V13 Matrix and the last best predictions
df_master = pl.read_parquet("/kaggle/working/master_features_v13.parquet").to_pandas()
# We use the V12_PSEUDO preds to find our new high-confidence targets
last_preds = pd.read_csv("/kaggle/working/DONE_WITHIT_V13_PSEUDO.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Advanced Pseudo-Labeling (Confidence Thresholding)
# Taking only the absolute extremes to anchor the model
confident_mules = last_preds[last_preds['is_mule'] > 0.998]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.002]['account_id']

print(f"Anchoring model with {len(confident_mules)} Silver-Bullet Mules from Test Set...")

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0

pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

# Combine for the Super-Dataset
df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)
df_test = df_test_orig.copy()

# 3. Features & Categoricals
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

# Fill and Type Cast
df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)
df_train[cat_features] = df_train[cat_features].fillna("UNKNOWN").astype(str).astype('category')
df_test[cat_features] = df_test[cat_features].fillna("UNKNOWN").astype(str).astype('category')

# 4. Train/Val Split (Validating ONLY on original ground truth)
X_val_orig = df_train_orig[features].fillna(-1)
for c in cat_features: X_val_orig[c] = X_val_orig[c].astype(str).astype('category')
y_val_orig = df_train_orig['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 5. Training the Tri-Force
print("\n[1/3] Training CatBoost...")
cat_model = CatBoostClassifier(
    iterations=2500, learning_rate=0.015, depth=8,
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=150, verbose=500
)
cat_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val), use_best_model=True)

print("\n[2/3] Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.02, max_depth=7,
    scale_pos_weight=15, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=100, eval_metric='auc', random_state=42
)
xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=500)

# 6. The Final Blend & Submission
print("\n[3/3] Generating Final Submission...")
cat_probs = cat_model.predict_proba(df_test[features])[:, 1]
xgb_probs = xgb_model.predict_proba(df_test[features])[:, 1]
final_probs = (cat_probs * 0.6) + (xgb_probs * 0.4)

# Build CSV with V13 Temporal Windows
sub_df = pd.DataFrame({'account_id': df_test['account_id'], 'is_mule': final_probs})
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()

# Rapidly re-extract temporal windows for the flagged set
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/DONE_WITHIT_V14_FINAL_PUSH.csv", index=False)
print("🏆 FINAL V14 SUBMISSION GENERATED!")

In [ ]:
import polars as pl
import os
import gc

print("--- ☢️ IGNITING LEVEL 18: INCREMENTAL CONTAGION ENGINE ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
V14_PREDS_PATH = "/kaggle/working/DONE_WITHIT_V14_FINAL_PUSH.csv"

# 1. Load Risk Scores into a small, fast lookup
risk_df = pl.read_csv(V14_PREDS_PATH).select([
    pl.col("account_id").alias("cp_id"), 
    pl.col("is_mule").alias("cp_risk")
])

batch_results = []

# 2. Process Batch by Batch (This is the secret to staying under 16GB RAM)
for i in range(1, 5):
    batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"Processing {batch_dir}...")
    
    # Process edges and internal logic for this batch only
    batch_lazy = (
        pl.scan_parquet(f"{batch_dir}/*.parquet")
        .select(["account_id", "counterparty_id", "amount", "txn_type"])
        .with_columns([
            pl.col("amount").abs().cast(pl.String).str.extract(r"([1-9])").cast(pl.Int32).alias("first_digit"),
            pl.when(pl.col("txn_type").str.contains("C")).then(1).otherwise(0).alias("txn_binary")
        ])
    )
    
    # Join with risk and aggregate
    batch_agg = (
        batch_lazy.join(risk_df.lazy(), left_on="counterparty_id", right_on="cp_id", how="left")
        .with_columns(pl.col("cp_risk").fill_null(0.1))
        .group_by("account_id")
        .agg([
            pl.col("cp_risk").mean().alias("neighbor_avg_risk"),
            (pl.col("first_digit").is_in([1, 2]).sum() / pl.len()).alias("benford_ratio"),
            pl.col("txn_binary").diff().abs().mean().alias("seq_stability")
        ])
        .collect() # Collect only the summary (small)
    )
    
    batch_results.append(batch_agg)
    gc.collect()

# 3. Final Merge of Summaries
print("Merging Batch Summaries...")
final_internal_stats = (
    pl.concat(batch_results)
    .group_by("account_id")
    .agg([
        pl.col("neighbor_avg_risk").mean(),
        pl.col("benford_ratio").mean().alias("benford_score"),
        pl.col("seq_stability").mean().alias("collusion_index")
    ])
)

# 4. Final Join with V13 Master
master_v13 = pl.read_parquet("/kaggle/working/master_features_v13.parquet")
master_v14 = master_v13.join(final_internal_stats, on="account_id", how="left")

master_v14.write_parquet("/kaggle/working/master_features_v14.parquet")
print("✅ V14 MATRIX SAVED! RAM survived.")

In [ ]:
import polars as pl

print("--- ⚖️ IGNITING LEVEL 19: THE AML MASTER-RATIO INJECTION (V5) ---")

# 1. Load V14 Eagerly
v15_df = pl.read_parquet("/kaggle/working/master_features_v14.parquet")

# 2. Map exactly to the columns found in your printout
# We use 'global_debit' and 'global_credit' which are clearly in your list!
v15_df = v15_df.with_columns([
    # Flow Ratios: Rapid money movement
    (pl.col("global_debit") / (pl.col("global_credit") + 1e-9)).alias("credit_to_debit_ratio"),
    (pl.col("global_debit") - pl.col("global_credit")).abs().alias("net_flow_imbalance"),
    
    # Balance Utilization: The "High Flow, Low Balance" Mule signature
    # Using 'total_txn_volume' for a more robust turnover metric
    (pl.col("total_txn_volume") / (pl.col("avg_balance").abs() + 1)).alias("balance_turnover_rate"),
    (pl.col("total_tx_count") / (pl.col("avg_balance").abs() + 1)).alias("tx_density_per_rupee"),
    
    # Counterparty Intensity
    (pl.col("total_tx_count") / (pl.col("unique_counterparties") + 1)).alias("tx_per_counterparty_ratio")
])

# 3. Forging Interaction & Temporal IoU Combos
v15_df = v15_df.with_columns([
    (pl.col("burst_ratio") * pl.col("unique_counterparties")).alias("burst_counterparty_intensity"),
    (pl.col("wash_pass_through_ratio") * pl.col("burst_ratio")).alias("mule_logic_combo")
])

# 4. Peer/Branch Deviations (The "Relative" Radar)
print("Calculating Branch-Level Deviations...")
branch_stats = v15_df.group_by("branch_code").agg([
    pl.col("total_tx_count").mean().alias("branch_avg_tx_count"),
    pl.col("total_txn_volume").mean().alias("branch_avg_vol")
])

v15_df = v15_df.join(branch_stats, on="branch_code", how="left").with_columns([
    (pl.col("total_tx_count") / (pl.col("branch_avg_tx_count") + 1)).alias("tx_count_branch_deviation_v2"),
    (pl.col("total_txn_volume") / (pl.col("branch_avg_vol") + 1)).alias("vol_branch_deviation_v2")
]).drop(["branch_avg_tx_count", "branch_avg_vol"])

# 5. Save the 150+ Feature Podium Matrix
v15_df.write_parquet("/kaggle/working/master_features_v15.parquet")
print(f"✅ V15 MASTER MATRIX SAVED! Total Columns: {v15_df.shape[1]}")

In [ ]:
import polars as pl
import numpy as np
from sklearn.ensemble import IsolationForest

print("--- 🏆 IGNITING LEVEL 20: THE SUDH-STYLE MASTER INJECTION ---")

# 1. Load V15 Matrix
v15_df = pl.read_parquet("/kaggle/working/master_features_v15.parquet")

# 2. Structural & Graph-Lite Features (Star 1 & 5)
print("Calculating Entropy & Concentration...")
v20_df = v15_df.with_columns([
    # Counterparty Concentration: If one person gets 90% of the money
    (pl.col("max_cpty_concentration") / (pl.col("unique_counterparties") + 1)).alias("cpty_dominance_ratio"),
    
    # Entropy Approximation: Measure of 'Randomness' in counterparties
    (pl.col("unique_counterparties") / (pl.col("total_tx_count") + 1)).alias("cpty_entropy_proxy"),
    
    # Fan-In/Fan-Out Structural logic
    (pl.col("in_degree").fill_null(0) / (pl.col("out_degree").fill_null(0) + 1)).alias("fan_structural_imbalance"),
    (pl.col("total_tx_count") / (pl.col("unique_counterparties") + 1)).alias("repeat_transaction_ratio")
])

# 3. Burst x Counterparty Combos (Star 3 & 7)
print("Forging Late-Game Combo Interactions...")
v20_df = v20_df.with_columns([
    (pl.col("burst_ratio") * pl.col("unique_counterparties")).alias("burst_fan_out_intensity"),
    (pl.col("micro_ping_count") * pl.col("unique_counterparties")).alias("ping_network_reach"),
    (pl.col("balance_volatility") * pl.col("burst_ratio")).alias("volatility_burst_synergy"),
    (pl.col("wash_pass_through_ratio") * pl.col("unique_counterparties")).alias("relay_node_score")
])

# 4. Anomaly Features: Isolation Forest (Star 6)
# We use a subset of the strongest features to train a fast anomaly detector
print("Dropping the Anomaly Nuke (Isolation Forest)...")
anomaly_cols = [
    "burst_ratio", "neighbor_avg_risk", "balance_turnover_rate", 
    "cpty_entropy_proxy", "collusion_index"
]
# Convert to numpy for Sklearn, fill NAs for the forest
X_anomaly = v20_df.select(anomaly_cols).fill_null(-1).to_numpy()

# Contamination set to 5% (assuming roughly 5% are mules)
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
v20_df = v20_df.with_columns([
    pl.Series(iso.fit_predict(X_anomaly)).alias("iso_forest_anomaly_raw")
])

# 5. Final Peer Deviations (Star 4)
print("Final Peer Deviation Mapping...")
v20_df = v20_df.with_columns([
    (pl.col("avg_balance") / (pl.col("overall_mean_balance") + 1)).alias("peer_balance_deviation"),
    (pl.col("total_products") / (pl.col("total_products").mean() + 1)).alias("product_density_ratio")
])

# 6. Save the 160+ Feature Matrix (THE FINAL DATASET)
v20_df.write_parquet("/kaggle/working/master_features_v20_FINAL.parquet")
print(f"✅ V20 FINAL MATRIX SAVED! Shape: {v20_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

print("--- 🥇 THE RANK 1 FINAL PUSH: V15 TRI-FORCE ---")

# 1. Load the Final V20 Matrix
df_master = pl.read_parquet("/kaggle/working/master_features_v20_FINAL.parquet").to_pandas()
# Use our previous V14 Pseudo-labels for consistency
last_preds = pd.read_csv("/kaggle/working/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Refined Pseudo-Labeling (Anchoring the extremes)
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

print(f"Adding {len(confident_mules)} 'Atomic' Mules to Training...")

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)

# 3. Clean and Prep
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region', 'branch_city', 'branch_state'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

# 4. Train/Val Split (Validate ONLY on ground truth)
X_val_orig = df_train_orig[features].fillna(-1)
for c in cat_features: X_val_orig[c] = X_val_orig[c].astype(str).astype('category')
y_val_orig = df_train_orig['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 5. Training the Heavy Hitters
print("\n[1/3] Training CatBoost Podium...")
cat_model = CatBoostClassifier(
    iterations=3000, learning_rate=0.012, depth=8,
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=200, verbose=500
)
cat_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val), use_best_model=True)

print("\n[2/3] Training XGBoost Podium...")
xgb_model = xgb.XGBClassifier(
    n_estimators=2500, learning_rate=0.015, max_depth=7,
    scale_pos_weight=15, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=150, eval_metric='auc', random_state=42
)
xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=500)

# 6. The 65/35 Gold Blend
cat_probs = cat_model.predict_proba(df_test_orig[features])[:, 1]
xgb_probs = xgb_model.predict_proba(df_test_orig[features])[:, 1]
final_probs = (cat_probs * 0.65) + (xgb_probs * 0.35)

# 7. Final Submission Assembly
sub_df = pd.DataFrame({'account_id': df_test_orig['account_id'], 'is_mule': final_probs})
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/FINAL_RANK_1_ATTEMPT.csv", index=False)
print("🏆 PODIUM SUBMISSION GENERATED! Upload FINAL_RANK_1_ATTEMPT.csv immediately.")

In [ ]:
import polars as pl
import gc

print("--- 💸 IGNITING LEVEL 21: THE STRUCTURING MODULO (TAX EVADER SIGNAL) ---")
DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

batch_results = []

# 1. Process Batch by Batch to prevent RAM blowout
for i in range(1, 5):
    batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"Hunting round numbers in {batch_dir}...")
    
    batch_lazy = (
        pl.scan_parquet(f"{batch_dir}/*.parquet")
        .select(["account_id", "amount"])
        .with_columns([
            pl.col("amount").abs().cast(pl.Float64).alias("abs_amt")
        ])
        .with_columns([
            # Calculate the remainder when dividing by 10k and 50k
            (pl.col("abs_amt") % 10000).alias("mod_10k"),
            (pl.col("abs_amt") % 50000).alias("mod_50k")
        ])
        .with_columns([
            # Check if the amount is within 10 rupees of a perfect boundary (e.g., 49,990 to 50,010)
            ((pl.col("mod_10k") <= 10) | (pl.col("mod_10k") >= 9990)).cast(pl.Int32).alias("is_structured_10k"),
            ((pl.col("mod_50k") <= 10) | (pl.col("mod_50k") >= 49990)).cast(pl.Int32).alias("is_structured_50k")
        ])
        .group_by("account_id")
        .agg([
            pl.col("is_structured_10k").sum().alias("structured_10k_count"),
            pl.col("is_structured_50k").sum().alias("structured_50k_count"),
            pl.len().alias("batch_tx_count") # Use pl.len() as requested by Polars earlier
        ])
        .collect()
    )
    
    batch_results.append(batch_lazy)
    gc.collect()

# 2. Forge the Modulo Master-Feature
print("Forging the Modulo Ratios...")
structuring_df = (
    pl.concat(batch_results)
    .group_by("account_id")
    .agg([
        pl.col("structured_10k_count").sum(),
        pl.col("structured_50k_count").sum(),
        pl.col("batch_tx_count").sum().alias("total_tx")
    ])
    .with_columns([
        # Convert raw counts to a ratio so high-volume accounts aren't unfairly penalized
        (pl.col("structured_10k_count") / (pl.col("total_tx") + 1)).alias("structured_10k_ratio"),
        (pl.col("structured_50k_count") / (pl.col("total_tx") + 1)).alias("structured_50k_ratio")
    ])
    .select(["account_id", "structured_10k_ratio", "structured_50k_ratio"])
)

# 3. Inject into our V20 Master Matrix
master_df = pl.read_parquet("/kaggle/working/master_features_v20_FINAL.parquet")
master_df = master_df.join(structuring_df, on="account_id", how="left").with_columns([
    pl.col("structured_10k_ratio").fill_null(0.0),
    pl.col("structured_50k_ratio").fill_null(0.0)
])

master_df.write_parquet("/kaggle/working/master_features_v21.parquet")
print(f"✅ V21 SAVED WITH MODULO MATH! Matrix Shape: {master_df.shape}")

In [ ]:
import polars as pl
import networkx as nx
import gc

print("--- 🕸️ IGNITING LEVEL 22: PAGERANK CENTRALITY (THE CARTEL HUB DETECTOR) ---")
DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

batch_edges = []

# 1. Extract Unique Edges (Connections) Batch by Batch
for i in range(1, 5):
    batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"Mapping network edges in {batch_dir}...")
    edges = (
        pl.scan_parquet(f"{batch_dir}/*.parquet")
        .select(["account_id", "counterparty_id"])
        .drop_nulls()
        .unique()
        .collect()
    )
    batch_edges.append(edges)

# 2. Forge the Global Network Edge List
print("Forging the Global Financial Graph...")
global_edges = pl.concat(batch_edges).unique().to_pandas()

del batch_edges
gc.collect()

# 3. Build the Directed Graph and Calculate Centrality
print("Initializing NetworkX DiGraph (Connecting the Cartel)...")
# Directed edge: account_id -> counterparty_id (Money Flow)
G = nx.from_pandas_edgelist(global_edges, source='account_id', target='counterparty_id', create_using=nx.DiGraph())

print("Executing PageRank Algorithm (This takes ~60 seconds)...")
# alpha=0.85 is standard. This calculates the global importance of every single account.
pr = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-06)

# Convert the results back to Polars
pr_df = pl.DataFrame({
    "account_id": list(pr.keys()),
    "pagerank_score": list(pr.values())
})

# Nuke the graph from RAM
del G, global_edges, pr
gc.collect()

# 4. Inject into the Master Matrix and Create the "Wash Node" Combo
print("Injecting Topology into V21 Master Matrix...")
master_df = pl.read_parquet("/kaggle/working/master_features_v21.parquet")

# Ensure account_id types match for a smooth join
if master_df.schema["account_id"] != pr_df.schema["account_id"]:
    pr_df = pr_df.with_columns(pl.col("account_id").cast(master_df.schema["account_id"]))

master_df = master_df.join(pr_df, on="account_id", how="left").with_columns(
    pl.col("pagerank_score").fill_null(0.0)
)

# GOD-TIER COMBO: The Master Wash Node Score
# High Centrality + High Pass-Through Ratio = Guaranteed Cartel Hub
if "wash_pass_through_ratio" in master_df.columns:
    master_df = master_df.with_columns(
        (pl.col("pagerank_score") * pl.col("wash_pass_through_ratio")).alias("master_wash_node_score")
    )
else:
    master_df = master_df.with_columns(
        (pl.col("pagerank_score") * pl.col("total_tx_count")).alias("master_wash_node_score")
    )

master_df.write_parquet("/kaggle/working/master_features_v22.parquet")
print(f"✅ V22 SAVED WITH PAGERANK TOPOLOGY! Matrix Shape: {master_df.shape}")

In [ ]:
import polars as pl
import gc

print("--- ⏱️ IGNITING LEVEL 23: TIME STOCHASTICITY (OOM-FIXED) ---")
DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

batch_results = []

# 1. Process Time Deltas Batch-by-Batch
for i in range(1, 5):
    batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"Extracting temporal signatures in {batch_dir}...")
    
    batch_lazy = (
        pl.scan_parquet(f"{batch_dir}/*.parquet")
        .select(["account_id", "transaction_timestamp"])
        .drop_nulls()
        .with_columns(
            # Convert to datetime safely
            pl.col("transaction_timestamp").str.to_datetime(strict=False)
        )
        .sort(["account_id", "transaction_timestamp"])
        .with_columns(
            # Calculate difference in seconds between consecutive transactions
            pl.col("transaction_timestamp").diff().dt.total_seconds().over("account_id").alias("delta")
        )
        .drop_nulls("delta") # Drop the first transaction per account in this batch (no delta)
    )
    
    # 2. Calculate the math components we need for Variance (Sum and Sum of Squares)
    batch_agg = (
        batch_lazy.group_by("account_id").agg([
            pl.col("delta").sum().alias("sum_delta"),
            (pl.col("delta") ** 2).sum().alias("sum_delta_sq"),
            pl.len().alias("count_delta")
        ])
        .collect()
    )
    
    batch_results.append(batch_agg)
    gc.collect()

# 3. Forge the Global Statistics mathematically
print("Forging the Global Stochasticity Metrics...")
stochastic_stats = (
    pl.concat(batch_results)
    .group_by("account_id")
    .agg([
        pl.col("sum_delta").sum(),
        pl.col("sum_delta_sq").sum(),
        pl.col("count_delta").sum()
    ])
    .filter(pl.col("count_delta") > 1) # Need at least 2 deltas to calculate variance
    .with_columns([
        (pl.col("sum_delta") / pl.col("count_delta")).alias("mean_delta")
    ])
    .with_columns([
        # Variance = E[X^2] - (E[X])^2
        ((pl.col("sum_delta_sq") / pl.col("count_delta")) - (pl.col("mean_delta") ** 2)).alias("var_delta")
    ])
    .with_columns([
        # Clip variance at 0 to prevent floating point errors, then take square root for Std Dev
        pl.col("var_delta").clip(lower_bound=0).sqrt().alias("std_delta")
    ])
    .with_columns([
        # Coefficient of Variation: std / mean
        (pl.col("std_delta") / (pl.col("mean_delta") + 1e-5)).alias("time_delta_cv")
    ])
    .select(["account_id", "time_delta_cv", "mean_delta", "std_delta"])
)

# 4. Inject into V22 Master Matrix
print("Injecting into V22 Master Matrix...")
master_df = pl.read_parquet("/kaggle/working/master_features_v22.parquet")

master_df = master_df.join(stochastic_stats, on="account_id", how="left").with_columns([
    pl.col("time_delta_cv").fill_null(-1.0), 
    pl.col("mean_delta").fill_null(-1.0),
    pl.col("std_delta").fill_null(-1.0)
])

# GOD-TIER COMBO: The Automation Index
# High volume with near-zero time variance = Automated Script
master_df = master_df.with_columns(
    (pl.col("total_tx_count") / (pl.col("time_delta_cv").clip(lower_bound=0.01))).alias("bot_automation_index")
)

master_df.write_parquet("/kaggle/working/master_features_v23.parquet")
print(f"✅ V23 SAVED WITH STOCHASTIC TIME DYNAMICS! Matrix Shape: {master_df.shape}")

In [ ]:
import polars as pl
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import gc

print("--- 🌌 IGNITING LEVEL 24: MANIFOLD COMPRESSION (THE 5D RADAR) ---")

# 1. Load V23
v23_df = pl.read_parquet("/kaggle/working/master_features_v23.parquet")
# Convert to pandas just for the scikit-learn compatibility
v23_pd = v23_df.to_pandas()

# 2. Isolate Numeric Features for the Math Engine
ignore_cols = [
    "account_id", "is_mule", "customer_id", "name", "account_status", 
    "branch_code", "product_code", "currency_code", "product_family",
    "account_opening_date", "last_mobile_update_date", "date_of_birth",
    "relationship_start_date", "freeze_date", "unfreeze_date",
    "last_kyc_date", "passbook_last_update_date", "branch_address",
    "branch_pin", "customer_pin", "cust_region", "branch_region",
    "branch_city", "branch_state", "gender", "address"
]

num_cols = v23_pd.select_dtypes(include=['number']).columns.tolist()
train_cols = [c for c in num_cols if c not in ignore_cols]

print(f"Compressing {len(train_cols)} numeric dimensions into 5 Meta-Coordinates...")

# Fill NaNs for Sklearn (Trees like -1, PCA needs continuous logic)
# We use the median for PCA to prevent extreme outliers from skewing the rotation
X = v23_pd[train_cols].fillna(v23_pd[train_cols].median()).values

# Scale the data (PCA requires everything to have mean=0, std=1 to work properly)
print("Standardizing Matrix Variance...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Fit PCA
print("Rotating Hyperspace (Running PCA)...")
pca = PCA(n_components=5, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Nuke the heavy arrays from RAM immediately
del X, X_scaled, v23_pd
gc.collect()

# 4. Inject back into the Polars Matrix
pca_df = pl.DataFrame({
    "account_id": v23_df["account_id"],
    "pca_meta_1": X_pca[:, 0],
    "pca_meta_2": X_pca[:, 1],
    "pca_meta_3": X_pca[:, 2],
    "pca_meta_4": X_pca[:, 3],
    "pca_meta_5": X_pca[:, 4],
})

master_df = v23_df.join(pca_df, on="account_id", how="left")

master_df.write_parquet("/kaggle/working/master_features_v24_FINAL.parquet")
print(f"✅ V24 SAVED! THE FINAL MATRIX IS LOCKED AT {master_df.shape[1]} FEATURES.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
import gc

print("--- 🥇 THE RANK 1 FINAL PUSH: V24 GOD-MODE ENSEMBLE ---")

# 1. Load the 176-Feature Behemoth
print("Loading V24 Master Matrix...")
df_master = pl.read_parquet("/kaggle/working/master_features_v24_FINAL.parquet").to_pandas()

# Load our previous V14 Predictions to use as Pseudo-Label anchors
last_preds = pd.read_csv("/kaggle/working/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Extreme Pseudo-Labeling (Anchoring the extremes to survive the Private Set)
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

print(f"Anchoring {len(confident_mules)} 'Atomic' Mules and {len(confident_legit)} 'Ironclad' Legit accounts...")

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

# Combine into the ultimate training set
df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)

# 3. Clean and Prep the Arena
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 
    'last_kyc_date', 'passbook_last_update_date', 
    'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code',
    'cust_region', 'branch_region', 'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"Training on {len(features)} engineered features...")

# Impute numericals with -1 (Trees learn this as 'Missing/Unknown' perfectly)
df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

# Format categoricals for CatBoost
for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

# 4. Train/Val Split (Validate ONLY on ground truth, not pseudo-labels)
X_val_orig = df_train_orig[features].fillna(-1)
for c in cat_features: X_val_orig[c] = X_val_orig[c].astype(str).astype('category')
y_val_orig = df_train_orig['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 5. Training the Heavy Hitters
print("\n[1/3] Igniting CatBoost (Deep Tree Architecture)...")
cat_model = CatBoostClassifier(
    iterations=3500, learning_rate=0.015, depth=8, # Slightly deeper/longer for 176 features
    eval_metric='AUC', auto_class_weights='Balanced',
    cat_features=cat_features, early_stopping_rounds=250, verbose=500, random_seed=42
)
cat_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val), use_best_model=True)

print("\n[2/3] Igniting XGBoost (Histogram Architecture)...")
xgb_model = xgb.XGBClassifier(
    n_estimators=3000, learning_rate=0.012, max_depth=7,
    scale_pos_weight=12, tree_method='hist', enable_categorical=True,
    early_stopping_rounds=200, eval_metric='auc', random_state=42
)
xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=500)

# 6. The Grandmaster Blend
print("\n[3/3] Forging the God-Mode Predictions...")
cat_probs = cat_model.predict_proba(df_test_orig[features])[:, 1]
xgb_probs = xgb_model.predict_proba(df_test_orig[features])[:, 1]
# We weight CatBoost slightly higher because it handles our PCA/Manifold and Categorical features better
final_probs = (cat_probs * 0.60) + (xgb_probs * 0.40)

# 7. Final Submission Assembly & Temporal IoU Extraction
sub_df = pd.DataFrame({'account_id': df_test_orig['account_id'], 'is_mule': final_probs})

# Threshold of 0.75 for pinpointing exactly when the mules were active
mule_accounts = sub_df[sub_df['is_mule'] > 0.75]['account_id'].tolist()
print(f"Extracting Suspicious Timestamps for {len(mule_accounts)} identified hubs...")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False))
    .group_by("account_id")
    .agg([
        pl.col("transaction_timestamp").min().alias("start_ts"),
        pl.col("transaction_timestamp").max().alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V24_RANK_1_GOD_MODE.csv", index=False)

print("\n🚀 DONE. V24_RANK_1_GOD_MODE.csv IS READY FOR DEPLOYMENT.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import gc

print("--- 💀 IGNITING LEVEL 26: RANK AVERAGING & THE TEMPORAL SNIPER (FULL RUN) ---")

# 1. Load the V24 Master Matrix
print("Loading V24 Master Matrix...")
df_master = pl.read_parquet("/kaggle/working/master_features_v24_FINAL.parquet").to_pandas()
last_preds = pd.read_csv("/kaggle/working/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Extreme Pseudo-Labeling (Anchors)
print("Setting up Pseudo-Label Anchors...")
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)

# 3. Clean and Prep
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

X_val_orig = df_train_orig[features].fillna(-1)
for c in cat_features: X_val_orig[c] = X_val_orig[c].astype(str).astype('category')
y_val_orig = df_train_orig['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 4. Train the Tri-Force (CatBoost, XGBoost, LightGBM)
print("\n[1/3] Igniting CatBoost...")
cat_model = CatBoostClassifier(iterations=3500, learning_rate=0.015, depth=8, eval_metric='AUC', auto_class_weights='Balanced', cat_features=cat_features, early_stopping_rounds=200, verbose=0, random_seed=42)
cat_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val))

print("[2/3] Igniting XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.012, max_depth=7, scale_pos_weight=12, tree_method='hist', enable_categorical=True, early_stopping_rounds=200, eval_metric='auc', random_state=42)
xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=0)

print("[3/3] Igniting LightGBM...")
X_train_lgb = df_train[features].copy()
X_val_lgb = X_val.copy()
X_test_lgb = df_test_orig[features].copy()
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_val_lgb[c] = X_val_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

lgb_model = lgb.LGBMClassifier(n_estimators=3500, learning_rate=0.01, num_leaves=63, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_lgb, df_train['is_mule'].astype(int), eval_set=[(X_val_lgb, y_val)], eval_metric='auc', callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)])

# 5. THE PUBLIC LB HACK: Rank Ensembling (BUG FIXED WITH .values)
print("\nExecuting Rank Percentile Ensembling...")
cat_probs = cat_model.predict_proba(df_test_orig[features])[:, 1]
xgb_probs = xgb_model.predict_proba(df_test_orig[features])[:, 1]
lgb_probs = lgb_model.predict_proba(X_test_lgb)[:, 1]

cat_ranks = pd.Series(cat_probs).rank(pct=True).values
xgb_ranks = pd.Series(xgb_probs).rank(pct=True).values
lgb_ranks = pd.Series(lgb_probs).rank(pct=True).values

final_ranks = (cat_ranks * 0.40) + (xgb_ranks * 0.30) + (lgb_ranks * 0.30)

sub_df = pd.DataFrame({
    'account_id': df_test_orig['account_id'].values, 
    'is_mule': final_ranks
})

# 6. THE TEMPORAL SNIPER: Dense Subgraph Extraction
mule_accounts = sub_df[sub_df['is_mule'] > 0.85]['account_id'].dropna().astype(str).tolist()
print(f"\nSniping Temporal Dense-Windows for {len(mule_accounts)} Cartel Hubs...")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(
        pl.col("transaction_timestamp").str.to_datetime(strict=False).dt.timestamp("ms").alias("ts_ms")
    )
    .group_by("account_id")
    .agg([
        pl.col("ts_ms").quantile(0.05).cast(pl.Datetime(time_unit="ms")).alias("start_ts"),
        pl.col("ts_ms").quantile(0.95).cast(pl.Datetime(time_unit="ms")).alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V26_TRIFORCE_RANK_SNIPER.csv", index=False)

print("🚀 DONE. V26_TRIFORCE_RANK_SNIPER.csv IS READY FOR DEPLOYMENT.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import gc

print("--- ☢️ IGNITING LEVEL 27: THE AUTOENCODER NUKE ---")

# 1. Load the Secured V24 Matrix
# (If your notebook restarted, update this path to your attached input dataset!)
DATA_PATH = "/kaggle/input/notebooks/tanmayistired/iitd-phase-2/master_features_v24_FINAL.parquet" 
print(f"Loading matrix from {DATA_PATH}...")
df_master = pl.read_parquet(DATA_PATH).to_pandas()

# Load Pseudo-labels from earlier
last_preds = pd.read_csv("/kaggle/input/notebooks/tanmayistired/iitd-phase-2/DONE_WITHIT_V14_FINAL_PUSH.csv")

# 2. Identify the "Pure" Legitimate Accounts
# We only want to train the AI on people we are 99.9% sure are innocent
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']
train_legit = df_master[(df_master['is_mule'] == 0) | (df_master['account_id'].isin(confident_legit))].copy()

# 3. Clean and Scale the Data for the Neural Network
# Neural Nets HATE missing values and unscaled data
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_master.columns if c not in drop_cols]
cat_features = df_master[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"Scaling {len(num_features)} continuous dimensions for Deep Learning...")

# Impute and Scale
scaler = RobustScaler() # RobustScaler ignores crazy outliers (which mules usually have)
X_legit_num = train_legit[num_features].fillna(train_legit[num_features].median())
X_legit_scaled = scaler.fit_transform(X_legit_num)

# Prepare the ENTIRE dataset to be pushed through the trained network later
X_all_num = df_master[num_features].fillna(df_master[num_features].median())
X_all_scaled = scaler.transform(X_all_num)

# 4. Architecting the AutoEncoder (The "Hourglass")
input_dim = X_legit_scaled.shape[1]

print("Forging the Hourglass Architecture...")
# Encoder
input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(64, activation='relu')(input_layer)
encoded = layers.BatchNormalization()(encoded)
encoded = layers.Dropout(0.2)(encoded)
encoded = layers.Dense(32, activation='relu')(encoded)
encoded = layers.Dense(8, activation='relu')(encoded) # The 8-Dimension Bottleneck

# Decoder
decoded = layers.Dense(32, activation='relu')(encoded)
decoded = layers.Dense(64, activation='relu')(decoded)
decoded = layers.BatchNormalization()(decoded)
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.002), loss='mse')

# 5. Training the AI on the Innocent
print("Training Neural Network on Legitimate DNA (Takes ~45 seconds)...")
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

autoencoder.fit(
    X_legit_scaled, X_legit_scaled, # X inputs to X outputs
    epochs=30,
    batch_size=512,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

# 6. The Execution: Forcing Everyone Through the Network
print("\nCalculating Cartel Reconstruction Error...")
reconstructions = autoencoder.predict(X_all_scaled, batch_size=1024)

# Calculate Mean Squared Error per account
mse = np.mean(np.power(X_all_scaled - reconstructions, 2), axis=1)

# Inject the God-Tier feature back into the Master Matrix
df_master['ae_reconstruction_loss'] = mse

print(f"Max Error (Likely a Cartel Boss): {mse.max():.2f}")
print(f"Median Error (Normal Citizen): {np.median(mse):.2f}")

# Save the V27 Matrix
df_master = pl.from_pandas(df_master)
df_master.write_parquet("/kaggle/working/master_features_v27_DEEP_LEARNING.parquet")
print("\n✅ V27 SECURED. Neural Network DNA injected!")

# Nuke arrays from RAM
del X_legit_num, X_legit_scaled, X_all_num, X_all_scaled, reconstructions
gc.collect()

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import gc

print("--- ⚔️ IGNITING LEVEL 28: THE AUTOENCODER TRI-FORCE ---")

# 1. Load the V27 Deep Learning Matrix
print("Loading V27 Master Matrix with Neural DNA...")
df_master = pl.read_parquet("/kaggle/working/master_features_v27_DEEP_LEARNING.parquet").to_pandas()

# Load our pseudo-labels
last_preds = pd.read_csv("/kaggle/input/notebooks/tanmayistired/iitd-phase-2/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Extreme Pseudo-Labeling
print("Setting up Pseudo-Label Anchors...")
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)

# 3. Clean and Prep
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"Training on {len(features)} dimensions (Including AE Reconstruction Loss)...")

df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

X_val_orig = df_train_orig[features].fillna(-1)
for c in cat_features: X_val_orig[c] = X_val_orig[c].astype(str).astype('category')
y_val_orig = df_train_orig['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 4. Train the Tri-Force
print("\n[1/3] Igniting CatBoost...")
cat_model = CatBoostClassifier(iterations=3500, learning_rate=0.015, depth=8, eval_metric='AUC', auto_class_weights='Balanced', cat_features=cat_features, early_stopping_rounds=200, verbose=0, random_seed=42)
cat_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val))

print("[2/3] Igniting XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.012, max_depth=7, scale_pos_weight=12, tree_method='hist', enable_categorical=True, early_stopping_rounds=200, eval_metric='auc', random_state=42)
xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=0)

print("[3/3] Igniting LightGBM...")
X_train_lgb = df_train[features].copy()
X_val_lgb = X_val.copy()
X_test_lgb = df_test_orig[features].copy()
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_val_lgb[c] = X_val_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

lgb_model = lgb.LGBMClassifier(n_estimators=3500, learning_rate=0.01, num_leaves=63, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_lgb, df_train['is_mule'].astype(int), eval_set=[(X_val_lgb, y_val)], eval_metric='auc', callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)])

# 5. Rank Ensembling
print("\nExecuting Rank Percentile Ensembling...")
cat_probs = cat_model.predict_proba(df_test_orig[features])[:, 1]
xgb_probs = xgb_model.predict_proba(df_test_orig[features])[:, 1]
lgb_probs = lgb_model.predict_proba(X_test_lgb)[:, 1]

cat_ranks = pd.Series(cat_probs).rank(pct=True).values
xgb_ranks = pd.Series(xgb_probs).rank(pct=True).values
lgb_ranks = pd.Series(lgb_probs).rank(pct=True).values

final_ranks = (cat_ranks * 0.40) + (xgb_ranks * 0.30) + (lgb_ranks * 0.30)

sub_df = pd.DataFrame({
    'account_id': df_test_orig['account_id'].values, 
    'is_mule': final_ranks
})

# 6. The Temporal Sniper (IoU Hack)
mule_accounts = sub_df[sub_df['is_mule'] > 0.85]['account_id'].dropna().astype(str).tolist()
print(f"\nSniping Temporal Dense-Windows for {len(mule_accounts)} Cartel Hubs...")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(
        pl.col("transaction_timestamp").str.to_datetime(strict=False).dt.timestamp("ms").alias("ts_ms")
    )
    .group_by("account_id")
    .agg([
        pl.col("ts_ms").quantile(0.05).cast(pl.Datetime(time_unit="ms")).alias("start_ts"),
        pl.col("ts_ms").quantile(0.95).cast(pl.Datetime(time_unit="ms")).alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V28_AUTOENCODER_TRIFORCE.csv", index=False)

print("🚀 DONE. V28_AUTOENCODER_TRIFORCE.csv IS READY TO ANNIHILATE THE LEADERBOARD.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import gc

print("--- 🛡️ IGNITING LEVEL 29: OOM-SAFE DEEP LEARNING ---")

# 1. Load Data
df_master = pl.read_parquet("/kaggle/working/master_features_v27_DEEP_LEARNING.parquet").to_pandas()
last_preds = pd.read_csv("/kaggle/input/notebooks/tanmayistired/iitd-phase-2/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train = df_master[df_master['is_mule'].notnull()].copy()
df_test = df_master[df_master['is_mule'].isnull()].copy()

# Add Pseudo Labels to Train
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

pseudo_mules = df_test[df_test['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test[df_test['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

df_train_full = pd.concat([df_train, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)
y_train_full = df_train_full['is_mule'].astype(int)

# 2. Isolate Numericals
cat_features = df_train_full.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
all_features = df_train_full.columns.tolist()
drop_cols = ['account_id', 'is_mule', 'customer_id', 'account_opening_date', 'last_mobile_update_date', 'date_of_birth', 'relationship_start_date', 'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 'customer_pin', 'branch_code', 'cust_region', 'branch_region', 'branch_city', 'branch_state', 'gender', 'address']
num_features = [c for c in all_features if c not in cat_features and c not in drop_cols]

print(f"Training DNN exclusively on {len(num_features)} continuous dimensions...")

X_train_num = df_train_full[num_features].fillna(df_train_full[num_features].median())
X_test_num = df_test[num_features].fillna(df_test[num_features].median())
test_ids = df_test['account_id'].values

# Nuke RAM immediately
del df_master, df_train, df_test, df_train_full, pseudo_mules, pseudo_legit
gc.collect()

# 3. Scale & Split
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

X_t, X_v, y_t, y_v = train_test_split(X_train_scaled, y_train_full, test_size=0.15, stratify=y_train_full, random_state=42)

# 4. Build & Train DNN
print("\nIgniting Deep Neural Network (DNN)...")
def build_dnn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='swish'),
        layers.BatchNormalization(),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

dnn_model = build_dnn(X_t.shape[1])
early_stop = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)

neg, pos = np.bincount(y_train_full)
class_weight = {0: (1 / neg) * (len(y_train_full) / 2.0), 1: (1 / pos) * (len(y_train_full) / 2.0)}

dnn_model.fit(X_t, y_t, validation_data=(X_v, y_v), epochs=60, batch_size=512, callbacks=[early_stop], class_weight=class_weight, verbose=1)

# 5. Extract DNN Ranks
print("\nExtracting Deep Learning Ranks...")
dnn_probs = dnn_model.predict(X_test_scaled, batch_size=1024).flatten()
dnn_ranks = pd.Series(dnn_probs).rank(pct=True).values

# 6. The Grandmaster Blend (Merging with V28)
print("Merging Deep Learning with V28 Tri-Force Trees...")
# We load the predictions from your successful V28 run!
v28_sub = pd.read_csv("/kaggle/working/V28_AUTOENCODER_TRIFORCE.csv")

# Ensure IDs match perfectly
v28_sub = v28_sub.set_index('account_id').loc[test_ids].reset_index()

# 80% Tree Tri-Force | 20% Neural Network
v28_sub['is_mule'] = (v28_sub['is_mule'] * 0.80) + (dnn_ranks * 0.20)

v28_sub.to_csv("/kaggle/working/V29_OOM_SAFE_QUADFORCE.csv", index=False)
print("🚀 DONE. V29_OOM_SAFE_QUADFORCE.csv IS READY.")

In [ ]:
import polars as pl
import pandas as pd
from sklearn.decomposition import TruncatedSVD
import gc

print("--- 🛠️ IGNITING LEVEL 30: OOM-SAFE NLP & SPATIAL FORGE ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

spatial_results = []
nlp_results = []

print("[1/3] Sweeping batches for Geographic & Behavioral Signatures...")
for i in range(1, 5):
    tx_batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    tx_add_batch_dir = f"{DATA_PATH}/transactions_additional/batch-{i}"
    
    # 1. OOM-Safe NLP Extraction (Polars Native Frequency Counting)
    print(f"  -> Extracting Channel Frequencies from Batch {i}...")
    nlp_batch = (
        pl.scan_parquet(f"{tx_batch_dir}/*.parquet")
        .select(["account_id", "channel"])
        .drop_nulls()
        .group_by(["account_id", "channel"])
        .agg(pl.len().alias("count"))
        .collect()
    )
    nlp_results.append(nlp_batch)
    
    # 2. Spatial Data Extraction
    print(f"  -> Extracting Spatial DNA from Batch {i}...")
    tx_base = pl.scan_parquet(f"{tx_batch_dir}/*.parquet").select(["transaction_id", "account_id"])
    spatial_batch = pl.scan_parquet(f"{tx_add_batch_dir}/*.parquet").select(["transaction_id", "latitude", "longitude"]).drop_nulls()
    
    spatial_joined = (
        tx_base.join(spatial_batch, on="transaction_id", how="inner")
        .group_by("account_id")
        .agg([
            pl.col("latitude").var().alias("lat_variance"),
            pl.col("longitude").var().alias("lon_variance"),
            (pl.col("latitude").max() - pl.col("latitude").min()).alias("lat_jump_max"),
            (pl.col("longitude").max() - pl.col("longitude").min()).alias("lon_jump_max")
        ])
        .collect()
    )
    spatial_results.append(spatial_joined)
    gc.collect()

print("\n[2/3] Forging Final Metrics (Bypassing Pandas RAM Trap)...")

# 1. Geographic Anomaly Consolidation
print("  -> Calculating Spatial Schizophrenia Metrics...")
spatial_df = pl.concat(spatial_results).group_by("account_id").agg([
    pl.col("lat_variance").max().fill_null(0.0),
    pl.col("lon_variance").max().fill_null(0.0),
    pl.col("lat_jump_max").max().fill_null(0.0),
    pl.col("lon_jump_max").max().fill_null(0.0)
]).with_columns(
    (pl.col("lat_variance") + pl.col("lon_variance")).alias("total_spatial_variance")
).to_pandas()

# 2. The NLP Fix: Pivot Matrix instead of String Concat
print("  -> Building Mathematical Channel Grid...")
nlp_concat = pl.concat(nlp_results).group_by(["account_id", "channel"]).agg(pl.col("count").sum())

# Pivot into an (Accounts x Channels) numerical matrix (Blazing fast in Polars)
nlp_pivot = nlp_concat.pivot(index="account_id", columns="channel", values="count", aggregate_function="sum").fill_null(0).to_pandas()

print("  -> Compressing Behavioral Matrix via SVD (The NLP Coordinates)...")
# Isolate just the channel count columns (Drop account_id for math)
channel_cols = [c for c in nlp_pivot.columns if c != "account_id"]
X_channels = nlp_pivot[channel_cols].values

# Compress the ~35 channel dimensions into 3 pure behavioral signatures
svd = TruncatedSVD(n_components=3, random_state=42)
svd_matrix = svd.fit_transform(X_channels)

nlp_df = pd.DataFrame({
    'account_id': nlp_pivot['account_id'],
    'behavior_nlp_dim1': svd_matrix[:, 0],
    'behavior_nlp_dim2': svd_matrix[:, 1],
    'behavior_nlp_dim3': svd_matrix[:, 2]
})

print("\n[3/3] Injecting into the Deep Learning Matrix...")
master_v27 = pl.read_parquet("/kaggle/working/master_features_v27_DEEP_LEARNING.parquet").to_pandas()

# Merge the new elite weapons
master_v30 = master_v27.merge(spatial_df, on="account_id", how="left")
master_v30 = master_v30.merge(nlp_df, on="account_id", how="left")

# Fill missing values for legitimate accounts that had no spatial/channel data
new_cols = ['lat_variance', 'lon_variance', 'lat_jump_max', 'lon_jump_max', 'total_spatial_variance', 'behavior_nlp_dim1', 'behavior_nlp_dim2', 'behavior_nlp_dim3']
master_v30[new_cols] = master_v30[new_cols].fillna(0.0)

# Save the Ultimate V30 Matrix
master_v30_pl = pl.from_pandas(master_v30)
master_v30_pl.write_parquet("/kaggle/working/master_features_v30_ULTIMATE.parquet")

print(f"✅ V30 SECURED! The matrix now has {master_v30.shape[1]} ultra-elite features.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np

print("--- 🛠️ IGNITING LEVEL 31: THE DEMOGRAPHIC INQUISITION ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

# 1. Load the V30 Ultimate Matrix
print("[1/3] Loading V30 Matrix...")
df_v30 = pl.read_parquet("/kaggle/working/master_features_v30_ULTIMATE.parquet")

# 2. Time & Demographic Math (Burner Velocity & Age Mismatch)
print("[2/3] Calculating Lifetime Velocity and Age Discrepancies...")

# Assuming the dataset ends around mid-2025 based on the challenge description
END_DATE = pl.lit(pd.to_datetime("2025-06-30"))

df_v30 = df_v30.with_columns([
    # Calculate Days Alive
    (END_DATE - pl.col("account_opening_date").str.to_datetime(strict=False)).dt.total_days().alias("tenure_days"),
    
    # Calculate Age in Years
    ((END_DATE - pl.col("date_of_birth").str.to_datetime(strict=False)).dt.total_days() / 365.25).alias("age_years")
])

# Protect against division by zero with .clip()
df_v30 = df_v30.with_columns([
    # Burner Velocity: Transactions per day alive
    (pl.col("total_tx_count") / pl.col("tenure_days").clip(lower_bound=1.0)).alias("lifetime_tx_velocity"),
    
    # Age-to-Volume Mismatch (Assuming you have a 'total_tx_count' or similar volume metric)
    (pl.col("total_tx_count") / pl.col("age_years").clip(lower_bound=18.0)).alias("tx_to_age_ratio")
])

# 3. The Ghost Footprint (Product Deficit)
print("[3/3] Hunting Financial Ghosts (Cross-referencing Product Details)...")

# Load linkage to map accounts to customers
linkage = pl.scan_parquet(f"{DATA_PATH}/customer_account_linkage.parquet").select(["account_id", "customer_id"])

# Load product details (loans, CCs, etc.)
products = pl.scan_parquet(f"{DATA_PATH}/product_details.parquet").select([
    "customer_id", "loan_count", "cc_count", "od_count"
])

# Join them together
ghost_tracker = (
    linkage.join(products, on="customer_id", how="left")
    .group_by("account_id")
    .agg([
        pl.col("loan_count").sum().fill_null(0).alias("total_loans"),
        pl.col("cc_count").sum().fill_null(0).alias("total_ccs"),
        pl.col("od_count").sum().fill_null(0).alias("total_ods")
    ])
    .with_columns([
        # If all 3 are zero, this is a Ghost Footprint (High Mule Risk)
        (pl.col("total_loans") + pl.col("total_ccs") + pl.col("total_ods")).alias("total_auxiliary_products")
    ])
    .collect()
)

# Inject into the Master Matrix
df_v31 = df_v30.join(ghost_tracker, on="account_id", how="left").with_columns(
    pl.col("total_auxiliary_products").fill_null(0) # Fill unlinked accounts with 0
)

# Create the Ultimate "Burner Ghost" Combo Feature
df_v31 = df_v31.with_columns(
    (pl.col("lifetime_tx_velocity") / (pl.col("total_auxiliary_products") + 1)).alias("ghost_velocity_index")
)

# Save the V31 Matrix
df_v31.write_parquet("/kaggle/working/master_features_v31_GOD_TIER.parquet")

print(f"✅ V31 SECURED! Matrix Shape: {df_v31.shape}")
print("New Weapons: 'lifetime_tx_velocity', 'tx_to_age_ratio', 'ghost_velocity_index'")

In [ ]:
import polars as pl
import pandas as pd
import gc

print("--- 🔬 IGNITING LEVEL 32: BEHAVIORAL PHYSICS & POLYNOMIAL CROSSES ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

# 1. Temporal Physics (Vampire & Salary Cycles)
print("[1/3] Sweeping batches for Temporal Physics (Vampire & Salary Cycles)...")
temporal_results = []

for i in range(1, 5):
    tx_batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"  -> Extracting Temporal DNA from Batch {i}...")
    
    batch_lazy = (
        pl.scan_parquet(f"{tx_batch_dir}/*.parquet")
        .select(["account_id", "transaction_timestamp"])
        .drop_nulls()
        .with_columns([
            pl.col("transaction_timestamp").str.to_datetime(strict=False).alias("ts")
        ])
        .with_columns([
            pl.col("ts").dt.hour().alias("hour"),
            pl.col("ts").dt.day().alias("day")
        ])
        .group_by("account_id")
        .agg([
            pl.len().alias("total_tx"),
            # The Vampire Index: Transactions strictly between Midnight and 5:59 AM
            pl.col("hour").filter(pl.col("hour") < 6).len().alias("vampire_tx_count"),
            # Salary Window: The end of the month (28th-31st) and start (1st-5th)
            pl.col("day").filter((pl.col("day") >= 28) | (pl.col("day") <= 5)).len().alias("salary_tx_count"),
            # The Dead Window: The middle of the month (10th-25th)
            pl.col("day").filter((pl.col("day") >= 10) & (pl.col("day") <= 25)).len().alias("dead_tx_count")
        ])
        .collect()
    )
    temporal_results.append(batch_lazy)
    gc.collect()

print("\n[2/3] Consolidating Global Temporal Physics...")
temporal_df = (
    pl.concat(temporal_results)
    .group_by("account_id")
    .agg([
        pl.col("total_tx").sum(),
        pl.col("vampire_tx_count").sum(),
        pl.col("salary_tx_count").sum(),
        pl.col("dead_tx_count").sum()
    ])
    .with_columns([
        # Percentage of total volume done while India is sleeping
        (pl.col("vampire_tx_count") / pl.col("total_tx").clip(lower_bound=1)).alias("vampire_ratio"),
        # Ratio of legitimate salary volume vs dormant volume
        (pl.col("salary_tx_count") / (pl.col("dead_tx_count") + 1)).alias("salary_to_dead_ratio")
    ])
    .select(["account_id", "vampire_ratio", "salary_to_dead_ratio"])
)

print("\n[3/3] Loading V31 Matrix and Injecting Level 32 Weapons...")
df_v31 = pl.read_parquet("/kaggle/working/master_features_v31_GOD_TIER.parquet")

# Merge Temporal Features
df_v32 = df_v31.join(temporal_df, on="account_id", how="left").with_columns([
    pl.col("vampire_ratio").fill_null(0.0),
    pl.col("salary_to_dead_ratio").fill_null(0.0)
])

# 2. POLYNOMIAL CROSSES (The Kaggle Nuke)
print("  -> Forging Polynomial Feature Crosses...")

cols = df_v32.columns

# Cross 1: Centralized Network Hub that behaves like a Robot
if "pagerank_score" in cols and "bot_automation_index" in cols:
    df_v32 = df_v32.with_columns((pl.col("pagerank_score") * pl.col("bot_automation_index")).alias("cross_pagerank_x_bot"))

# Cross 2: A financial Ghost (0 loans/CCs) turning over their balance at lightning speed
if "ghost_velocity_index" in cols and "balance_turnover_rate" in cols:
    df_v32 = df_v32.with_columns((pl.col("ghost_velocity_index") * pl.col("balance_turnover_rate")).alias("cross_ghost_x_turnover"))

# Cross 3: PCA Anomaly clustering specifically interacting with Night-Owl behavior
if "pca_meta_1" in cols and "vampire_ratio" in cols:
    df_v32 = df_v32.with_columns((pl.col("pca_meta_1") * pl.col("vampire_ratio")).alias("cross_pca1_x_vampire"))

# Cross 4: Deep Learning failure rate crossed with Graph Contagion Risk
if "ae_reconstruction_loss" in cols and "neighbor_avg_risk" in cols:
    df_v32 = df_v32.with_columns((pl.col("ae_reconstruction_loss") * pl.col("neighbor_avg_risk")).alias("cross_ae_x_contagion"))

df_v32.write_parquet("/kaggle/working/master_features_v32_BEHAVIORAL.parquet")

print(f"✅ V32 SECURED! Matrix Shape: {df_v32.shape}")
print("New Weapons Added: 'vampire_ratio', 'salary_to_dead_ratio', and 4 God-Tier Polynomial Crosses!")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import gc

print("--- ☢️ IGNITING LEVEL 33: THE V32 GOD-MODE QUAD-FORCE ---")

# 1. Load the 200-Feature Matrix
print("[1/5] Loading V32 Ultimate Matrix...")
df_master = pl.read_parquet("/kaggle/working/master_features_v32_BEHAVIORAL.parquet").to_pandas()
last_preds = pd.read_csv("/kaggle/input/notebooks/tanmayistired/iitd-phase-2/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Extreme Pseudo-Label Anchors (From the pure V14 baseline)
print("[2/5] Setting up Pure Anchors...")
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)

# Free RAM
del df_master, df_train_orig, pseudo_mules, pseudo_legit
gc.collect()

# 3. Clean and Prep the 200 Dimensions
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"Training on exactly {len(features)} engineered features...")

# Safely impute numericals and categoricals separately
df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

# THE FIX: df_train is already perfectly clean and categorized. Just copy it directly.
X_val_orig = df_train[features].copy()
y_val_orig = df_train['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 4. Train the Tri-Force (Trees)
print("\n[3/5] Igniting Tree Ensembles...")
print("  -> Training CatBoost...")
cat_model = CatBoostClassifier(iterations=3500, learning_rate=0.015, depth=8, eval_metric='AUC', auto_class_weights='Balanced', cat_features=cat_features, early_stopping_rounds=200, verbose=0, random_seed=42)
cat_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val))
cat_probs = cat_model.predict_proba(df_test_orig[features])[:, 1]
del cat_model; gc.collect()

print("  -> Training XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.012, max_depth=7, scale_pos_weight=12, tree_method='hist', enable_categorical=True, early_stopping_rounds=200, eval_metric='auc', random_state=42)
xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=0)
xgb_probs = xgb_model.predict_proba(df_test_orig[features])[:, 1]
del xgb_model; gc.collect()

print("  -> Training LightGBM...")
X_train_lgb = df_train[features].copy()
X_test_lgb = df_test_orig[features].copy()
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

lgb_model = lgb.LGBMClassifier(n_estimators=3500, learning_rate=0.01, num_leaves=63, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_lgb, df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], eval_metric='auc', callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)])
lgb_probs = lgb_model.predict_proba(X_test_lgb)[:, 1]
del lgb_model, X_train_lgb, X_test_lgb; gc.collect()

# 5. Train the Deep Neural Network (OOM Safe Continuous Features Only)
print("\n[4/5] Igniting Supervised Deep Neural Network...")
scaler = StandardScaler()
X_train_nn_scaled = scaler.fit_transform(df_train[num_features])
X_test_nn_scaled = scaler.transform(df_test_orig[num_features])

X_t_nn, X_v_nn, y_t_nn, y_v_nn = train_test_split(X_train_nn_scaled, df_train['is_mule'].astype(int), test_size=0.15, stratify=df_train['is_mule'].astype(int), random_state=42)

def build_dnn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='swish'),
        layers.BatchNormalization(),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

dnn_model = build_dnn(X_t_nn.shape[1])
early_stop = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)

neg, pos = np.bincount(df_train['is_mule'].astype(int))
class_weight = {0: (1 / neg) * (len(df_train) / 2.0), 1: (1 / pos) * (len(df_train) / 2.0)}

dnn_model.fit(X_t_nn, y_t_nn, validation_data=(X_v_nn, y_v_nn), epochs=60, batch_size=512, callbacks=[early_stop], class_weight=class_weight, verbose=0)
dnn_probs = dnn_model.predict(X_test_nn_scaled, batch_size=1024).flatten()
del dnn_model, X_train_nn_scaled, X_test_nn_scaled, X_t_nn, X_v_nn; gc.collect()

# 6. The Quad-Force Rank Ensemble
print("\n[5/5] Executing Quad-Force Rank Ensembling & Temporal Sniping...")
cat_ranks = pd.Series(cat_probs).rank(pct=True).values
xgb_ranks = pd.Series(xgb_probs).rank(pct=True).values
lgb_ranks = pd.Series(lgb_probs).rank(pct=True).values
dnn_ranks = pd.Series(dnn_probs).rank(pct=True).values

# 30% CatBoost | 25% XGBoost | 25% LightGBM | 20% DNN
final_ranks = (cat_ranks * 0.30) + (xgb_ranks * 0.25) + (lgb_ranks * 0.25) + (dnn_ranks * 0.20)
sub_df = pd.DataFrame({'account_id': df_test_orig['account_id'].values, 'is_mule': final_ranks})

# 7. The Temporal Sniper
mule_accounts = sub_df[sub_df['is_mule'] > 0.85]['account_id'].dropna().astype(str).tolist()

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False).dt.timestamp("ms").alias("ts_ms"))
    .group_by("account_id")
    .agg([
        pl.col("ts_ms").quantile(0.05).cast(pl.Datetime(time_unit="ms")).alias("start_ts"),
        pl.col("ts_ms").quantile(0.95).cast(pl.Datetime(time_unit="ms")).alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V33_GOD_MODE_QUADFORCE.csv", index=False)

print("🚀 DONE. V33_GOD_MODE_QUADFORCE.csv IS READY.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import gc

print("--- 🩸 IGNITING LEVEL 34: THE PURE-BLOOD QUAD-FORCE ---")

# 1. Load the 200-Feature Matrix
print("[1/5] Loading V32 Ultimate Matrix...")
df_master = pl.read_parquet("/kaggle/working/master_features_v32_BEHAVIORAL.parquet").to_pandas()

# 2. STRICT GROUND TRUTH ONLY (Zero Pseudo-Labels)
print("[2/5] Stripping Pseudo-Labels. Pure Ground Truth Only...")
df_train = df_master[df_master['is_mule'].notnull()].copy().reset_index(drop=True)
df_test_orig = df_master[df_master['is_mule'].isnull()].copy().reset_index(drop=True)

# Free RAM
del df_master
gc.collect()

# 3. Clean and Prep the 200 Dimensions
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"Training strictly on {len(features)} God-Tier engineered features...")

# Safely impute numericals
df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

# Safely impute categoricals
for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

X_val_orig = df_train[features].copy()
y_val_orig = df_train['is_mule'].astype(int)

# Stratified Split on the pure labels
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 4. Train the Tri-Force (Trees)
print("\n[3/5] Igniting Tree Ensembles...")
print("  -> Training CatBoost (Deep Architecture)...")
cat_model = CatBoostClassifier(iterations=3500, learning_rate=0.015, depth=8, eval_metric='AUC', auto_class_weights='Balanced', cat_features=cat_features, early_stopping_rounds=200, verbose=0, random_seed=42)
cat_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val))
cat_probs = cat_model.predict_proba(df_test_orig[features])[:, 1]
del cat_model; gc.collect()

print("  -> Training XGBoost (Histogram Architecture)...")
xgb_model = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.012, max_depth=7, scale_pos_weight=12, tree_method='hist', enable_categorical=True, early_stopping_rounds=200, eval_metric='auc', random_state=42)
xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=0)
xgb_probs = xgb_model.predict_proba(df_test_orig[features])[:, 1]
del xgb_model; gc.collect()

print("  -> Training LightGBM (Leaf-Wise Architecture)...")
X_train_lgb = df_train[features].copy()
X_test_lgb = df_test_orig[features].copy()
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

lgb_model = lgb.LGBMClassifier(n_estimators=3500, learning_rate=0.01, num_leaves=63, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_lgb, df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], eval_metric='auc', callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)])
lgb_probs = lgb_model.predict_proba(X_test_lgb)[:, 1]
del lgb_model, X_train_lgb, X_test_lgb; gc.collect()

# 5. Train the Deep Neural Network (OOM Safe Continuous Features Only)
print("\n[4/5] Igniting Supervised Deep Neural Network...")
scaler = StandardScaler()
X_train_nn_scaled = scaler.fit_transform(df_train[num_features])
X_test_nn_scaled = scaler.transform(df_test_orig[num_features])

X_t_nn, X_v_nn, y_t_nn, y_v_nn = train_test_split(X_train_nn_scaled, df_train['is_mule'].astype(int), test_size=0.15, stratify=df_train['is_mule'].astype(int), random_state=42)

def build_dnn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='swish'),
        layers.BatchNormalization(),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

dnn_model = build_dnn(X_t_nn.shape[1])
early_stop = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)

neg, pos = np.bincount(df_train['is_mule'].astype(int))
class_weight = {0: (1 / neg) * (len(df_train) / 2.0), 1: (1 / pos) * (len(df_train) / 2.0)}

dnn_model.fit(X_t_nn, y_t_nn, validation_data=(X_v_nn, y_v_nn), epochs=60, batch_size=512, callbacks=[early_stop], class_weight=class_weight, verbose=0)
dnn_probs = dnn_model.predict(X_test_nn_scaled, batch_size=1024).flatten()
del dnn_model, X_train_nn_scaled, X_test_nn_scaled, X_t_nn, X_v_nn; gc.collect()

# 6. The Quad-Force Rank Ensemble
print("\n[5/5] Executing Quad-Force Rank Ensembling & Temporal Sniping...")
cat_ranks = pd.Series(cat_probs).rank(pct=True).values
xgb_ranks = pd.Series(xgb_probs).rank(pct=True).values
lgb_ranks = pd.Series(lgb_probs).rank(pct=True).values
dnn_ranks = pd.Series(dnn_probs).rank(pct=True).values

# 30% CatBoost | 25% XGBoost | 25% LightGBM | 20% DNN
final_ranks = (cat_ranks * 0.30) + (xgb_ranks * 0.25) + (lgb_ranks * 0.25) + (dnn_ranks * 0.20)
sub_df = pd.DataFrame({'account_id': df_test_orig['account_id'].values, 'is_mule': final_ranks})

# 7. The Temporal Sniper
mule_accounts = sub_df[sub_df['is_mule'] > 0.85]['account_id'].dropna().astype(str).tolist()

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False).dt.timestamp("ms").alias("ts_ms"))
    .group_by("account_id")
    .agg([
        pl.col("ts_ms").quantile(0.05).cast(pl.Datetime(time_unit="ms")).alias("start_ts"),
        pl.col("ts_ms").quantile(0.95).cast(pl.Datetime(time_unit="ms")).alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V34_PURE_BLOOD_QUADFORCE.csv", index=False)

print("🚀 DONE. V34_PURE_BLOOD_QUADFORCE.csv IS READY. DESTROY THE LEADERBOARD.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import gc

print("--- 🧪 IGNITING LEVEL 35: FOCAL DISTILLATION & SOFT LABELS ---")

# 1. Load the 200-Feature Matrix & Teacher Predictions
print("[1/5] Loading V32 Ultimate Matrix and V14 Teacher...")
df_master = pl.read_parquet("/kaggle/working/master_features_v32_BEHAVIORAL.parquet").to_pandas()
v14_preds = pd.read_csv("/kaggle/input/notebooks/tanmayistired/iitd-phase-2/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train = df_master[df_master['is_mule'].notnull()].copy().reset_index(drop=True)
df_test = df_master[df_master['is_mule'].isnull()].copy().reset_index(drop=True)

del df_master; gc.collect()

# 2. THE GRANDMASTER TRICK: Soft Label Distillation
print("[2/5] Synthesizing Soft Targets (Knowledge Distillation)...")
# We align the V14 predictions with our training set
v14_train_preds = df_train.merge(v14_preds, on='account_id', how='left', suffixes=('', '_v14'))

# The Blend: 75% RBIH Ground Truth + 25% V14 Teacher Probability
# This smooths out the bank's false negatives without overriding them completely.
df_train['soft_label'] = (df_train['is_mule'] * 0.75) + (v14_train_preds['is_mule_v14'].fillna(0) * 0.25)

# 3. Clean and Prep
drop_cols = [
    'account_id', 'is_mule', 'soft_label', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

df_train[num_features] = df_train[num_features].fillna(-1)
df_test[num_features] = df_test[num_features].fillna(-1)

for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test[c] = df_test[c].fillna("UNKNOWN").astype(str).astype('category')

# We now split predicting the SOFT LABEL, not the hard binary label
X_val_orig = df_train[features].copy()
y_val_soft = df_train['soft_label'].values
# We stratify on the original hard label to maintain distribution
_, X_val, _, y_val_soft_split = train_test_split(X_val_orig, y_val_soft, test_size=0.15, stratify=df_train['is_mule'].astype(int), random_state=42)

# 4. Train the Tri-Force (Regression on Soft Targets)
print("\n[3/5] Igniting Tree Ensembles on Soft Probabilities...")

# CatBoost using RMSE to map the probability curve
print("  -> Training CatBoost Regressor...")
cat_model = CatBoostRegressor(iterations=3500, learning_rate=0.015, depth=8, eval_metric='RMSE', cat_features=cat_features, early_stopping_rounds=200, verbose=0, random_seed=42)
cat_model.fit(df_train[features], df_train['soft_label'], eval_set=(X_val, y_val_soft_split))
cat_probs = np.clip(cat_model.predict(df_test[features]), 0, 1) # Clip to valid probability bounds
del cat_model; gc.collect()

# XGBoost using custom scaling
print("  -> Training XGBoost Regressor...")
xgb_model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.012, max_depth=7, tree_method='hist', enable_categorical=True, early_stopping_rounds=200, eval_metric='rmse', random_state=42)
xgb_model.fit(df_train[features], df_train['soft_label'], eval_set=[(X_val, y_val_soft_split)], verbose=0)
xgb_probs = np.clip(xgb_model.predict(df_test[features]), 0, 1)
del xgb_model; gc.collect()

# LightGBM using Cross-Entropy on continuous targets (Focal emulation)
print("  -> Training LightGBM Regressor...")
X_train_lgb = df_train[features].copy()
X_test_lgb = df_test[features].copy()
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

lgb_model = lgb.LGBMRegressor(n_estimators=3500, learning_rate=0.01, num_leaves=63, max_depth=8, objective='regression', random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_lgb, df_train['soft_label'], eval_set=[(X_val, y_val_soft_split)], eval_metric='rmse', callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)])
lgb_probs = np.clip(lgb_model.predict(X_test_lgb), 0, 1)
del lgb_model, X_train_lgb, X_test_lgb; gc.collect()

# 5. Deep Learning (Keeping Binary Cross Entropy for raw contrast)
print("\n[4/5] Igniting Deep Neural Network...")
scaler = StandardScaler()
X_train_nn_scaled = scaler.fit_transform(df_train[num_features])
X_test_nn_scaled = scaler.transform(df_test[num_features])

X_t_nn, X_v_nn, y_t_nn, y_v_nn = train_test_split(X_train_nn_scaled, df_train['is_mule'].astype(int), test_size=0.15, stratify=df_train['is_mule'].astype(int), random_state=42)

def build_dnn(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='swish'),
        layers.BatchNormalization(),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

dnn_model = build_dnn(X_t_nn.shape[1])
early_stop = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)
neg, pos = np.bincount(df_train['is_mule'].astype(int))
class_weight = {0: (1 / neg) * (len(df_train) / 2.0), 1: (1 / pos) * (len(df_train) / 2.0)}
dnn_model.fit(X_t_nn, y_t_nn, validation_data=(X_v_nn, y_v_nn), epochs=60, batch_size=512, callbacks=[early_stop], class_weight=class_weight, verbose=0)
dnn_probs = dnn_model.predict(X_test_nn_scaled, batch_size=1024).flatten()
del dnn_model, X_train_nn_scaled, X_test_nn_scaled, X_t_nn, X_v_nn; gc.collect()

# 6. Rank Ensembling
print("\n[5/5] Executing Distilled Rank Ensembling...")
cat_ranks = pd.Series(cat_probs).rank(pct=True).values
xgb_ranks = pd.Series(xgb_probs).rank(pct=True).values
lgb_ranks = pd.Series(lgb_probs).rank(pct=True).values
dnn_ranks = pd.Series(dnn_probs).rank(pct=True).values

final_ranks = (cat_ranks * 0.30) + (xgb_ranks * 0.25) + (lgb_ranks * 0.25) + (dnn_ranks * 0.20)
sub_df = pd.DataFrame({'account_id': df_test['account_id'].values, 'is_mule': final_ranks})

# 7. Temporal Sniper (Re-aligned)
mule_accounts = sub_df[sub_df['is_mule'] > 0.85]['account_id'].dropna().astype(str).tolist()
DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False).dt.timestamp("ms").alias("ts_ms"))
    .group_by("account_id")
    .agg([
        pl.col("ts_ms").quantile(0.05).cast(pl.Datetime(time_unit="ms")).alias("start_ts"),
        pl.col("ts_ms").quantile(0.95).cast(pl.Datetime(time_unit="ms")).alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V35_FOCAL_DISTILLATION.csv", index=False)

print("🚀 DONE. V35_FOCAL_DISTILLATION.csv IS READY.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import gc

print("--- 🛡️ IGNITING LEVEL 37: OOM-SAFE VAULT INJECTION (FAIL-SAFE) ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

time_results = []
balance_results = []

print("[1/4] Sweeping batches for Temporal Gaps and Entropy...")
for i in range(1, 5):
    tx_batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"  -> Processing Time Physics for Batch {i}...")
    
    time_batch = (
        pl.scan_parquet(f"{tx_batch_dir}/*.parquet")
        .select(["account_id", "transaction_timestamp"])
        .drop_nulls()
        .with_columns(
            pl.col("transaction_timestamp").str.to_datetime(strict=False).alias("ts")
        )
        .with_columns([
            pl.col("ts").dt.hour().alias("hour"),
            pl.col("ts").dt.minute().alias("minute"),
            pl.col("ts").dt.second().alias("second"),
            pl.col("ts").dt.timestamp("ms").alias("ts_ms")
        ])
        .group_by("account_id")
        .agg([
            pl.col("ts_ms").sort().diff().mean().alias("mean_gap_ms"),
            pl.col("ts_ms").sort().diff().max().alias("max_gap_ms"),
            (pl.col("ts_ms").sort().diff() ** 2).sum().alias("sum_gap_sq_ms"),
            (pl.col("ts_ms").sort().diff().std() / (pl.col("ts_ms").sort().diff().mean() + 1.0)).alias("cv_gap"),
            
            pl.col("hour").n_unique().alias("hour_diversity"),
            pl.col("minute").n_unique().alias("minute_diversity"),
            pl.col("second").n_unique().alias("second_diversity")
        ])
        .collect()
    )
    time_results.append(time_batch)
    gc.collect()

print("\n[2/4] Sweeping batches for Balance Physics...")
for i in range(1, 5):
    tx_batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    tx_add_batch_dir = f"{DATA_PATH}/transactions_additional/batch-{i}"
    print(f"  -> Processing Balance Physics for Batch {i}...")
    
    base_id = pl.scan_parquet(f"{tx_batch_dir}/*.parquet").select(["transaction_id", "account_id"])
    add_bal = pl.scan_parquet(f"{tx_add_batch_dir}/*.parquet").select(["transaction_id", "balance_after_transaction"])
    
    bal_batch = (
        base_id.join(add_bal, on="transaction_id", how="inner")
        .group_by("account_id")
        .agg([
            (pl.col("balance_after_transaction") <= 0).sum().alias("zero_cross_rate"),
            # Fill standard deviation with 0 immediately for single-transaction accounts
            pl.col("balance_after_transaction").std().fill_null(0.0).alias("balance_volatility"),
            (pl.col("balance_after_transaction").max() - pl.col("balance_after_transaction").min()).alias("balance_range")
        ])
        .collect()
    )
    balance_results.append(bal_batch)
    gc.collect()

print("\n[3/4] Consolidating and Formatting Features...")

time_df = pl.concat(time_results).group_by("account_id").agg([
    (pl.col("mean_gap_ms").mean() / 3600000).fill_null(0).alias("mean_gap_hours"),
    (pl.col("max_gap_ms").max() / 3600000).fill_null(0).alias("max_gap_hours"),
    (pl.col("sum_gap_sq_ms").sum() / (3600000**2)).fill_null(0).alias("sum_gap_sq"),
    pl.col("cv_gap").mean().fill_null(0).alias("cv_gap"),
    pl.col("hour_diversity").max().fill_null(1).alias("hour_diversity"),
    pl.col("minute_diversity").max().fill_null(1).alias("minute_diversity"),
    pl.col("second_diversity").max().fill_null(1).alias("second_diversity")
]).to_pandas()

time_df['hour_entropy'] = np.log1p(time_df['hour_diversity'])
time_df['minute_entropy'] = np.log1p(time_df['minute_diversity'])
time_df['second_entropy'] = np.log1p(time_df['second_diversity'])
time_df = time_df.drop(columns=['hour_diversity', 'minute_diversity', 'second_diversity'])
del time_results; gc.collect()

bal_df = pl.concat(balance_results).group_by("account_id").agg([
    pl.col("zero_cross_rate").sum().fill_null(0).alias("zero_cross_rate"),
    pl.col("balance_volatility").mean().fill_null(0).alias("balance_volatility"),
    pl.col("balance_range").max().fill_null(0).alias("balance_range")
]).to_pandas()
del balance_results; gc.collect()

print("\n[4/4] Injecting into the God-Tier Matrix...")
master_v32 = pl.read_parquet("/kaggle/working/master_features_v32_BEHAVIORAL.parquet").to_pandas()

master_v37 = master_v32.merge(time_df, on="account_id", how="left")
master_v37 = master_v37.merge(bal_df, on="account_id", how="left")

# THE FAIL-SAFE: Check if the column exists before filling. If it ghosted, create it.
new_cols = ['mean_gap_hours', 'max_gap_hours', 'sum_gap_sq', 'cv_gap', 'hour_entropy', 'minute_entropy', 'second_entropy', 'zero_cross_rate', 'balance_volatility', 'balance_range']

for col in new_cols:
    if col not in master_v37.columns:
        print(f"⚠️ Warning: '{col}' was ghosted by Polars. Injecting safe defaults.")
        master_v37[col] = 0.0
    else:
        master_v37[col] = master_v37[col].fillna(0.0)

master_v37_pl = pl.from_pandas(master_v37)
master_v37_pl.write_parquet("/kaggle/working/master_features_v37_VAULT.parquet")

print(f"✅ V37 SECURED! Matrix Shape: {master_v37.shape}")
print("Fail-Safes activated. 10 massive features successfully injected.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import gc

print("--- 🛡️ IGNITING LEVEL 38: THE WASH-PIPE INJECTION ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
flow_results = []

print("[1/3] Sweeping batches for Money Flow Physics (CR vs DR)...")
for i in range(1, 5):
    tx_batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"  -> Processing Flow Dynamics for Batch {i}...")
    
    flow_batch = (
        pl.scan_parquet(f"{tx_batch_dir}/*.parquet")
        .select(["account_id", "amount", "txn_type"])
        .drop_nulls()
        .group_by("account_id")
        .agg([
            # Sum of Credits (Money In)
            pl.col("amount").filter(pl.col("txn_type") == "CR").sum().alias("total_credit"),
            # Sum of Debits (Money Out)
            pl.col("amount").filter(pl.col("txn_type") == "DR").sum().alias("total_debit"),
            
            # Count of Credits vs Debits for Smurfing detection
            pl.col("amount").filter(pl.col("txn_type") == "CR").len().alias("count_credit"),
            pl.col("amount").filter(pl.col("txn_type") == "DR").len().alias("count_debit")
        ])
        .collect()
    )
    flow_results.append(flow_batch)
    gc.collect()

print("\n[2/3] Consolidating Flow Features...")
# If accounts span multiple batches, we sum their totals
flow_df = pl.concat(flow_results).group_by("account_id").agg([
    pl.col("total_credit").sum().fill_null(0.0).alias("total_credit"),
    pl.col("total_debit").sum().fill_null(0.0).alias("total_debit"),
    pl.col("count_credit").sum().fill_null(0).alias("count_credit"),
    pl.col("count_debit").sum().fill_null(0).alias("count_debit")
]).to_pandas()

# THE MATH: Calculate the advanced ratios safely
# 1. Flow-Through Ratio: How perfectly does Debit match Credit? 
# (Using min/max so it's always <= 1.0. Near 1.0 is highly suspicious for high volumes)
flow_df['flow_through_ratio'] = np.minimum(flow_df['total_credit'], flow_df['total_debit']) / \
                                (np.maximum(flow_df['total_credit'], flow_df['total_debit']) + 1.0)

# 2. Aggregation Ratio (Smurfing): Many small credits -> Few large debits
# High ratio means money is being aggregated. Low ratio means money is being dispersed.
flow_df['aggregation_ratio'] = flow_df['count_credit'] / (flow_df['count_debit'] + 1.0)

# 3. Balance Drain Ratio: Average Outflow Size vs Average Inflow Size
flow_df['avg_inflow'] = flow_df['total_credit'] / (flow_df['count_credit'] + 1.0)
flow_df['avg_outflow'] = flow_df['total_debit'] / (flow_df['count_debit'] + 1.0)
flow_df['balance_drain_intensity'] = flow_df['avg_outflow'] / (flow_df['avg_inflow'] + 1.0)

# Clean up memory
flow_df.drop(columns=['total_credit', 'total_debit', 'count_credit', 'count_debit', 'avg_inflow', 'avg_outflow'], inplace=True)
del flow_results; gc.collect()

print("\n[3/3] Injecting into the God-Tier Matrix...")
# Load V37
master_v37 = pl.read_parquet("/kaggle/working/master_features_v37_VAULT.parquet").to_pandas()

master_v38 = master_v37.merge(flow_df, on="account_id", how="left")

# FAIL-SAFE: Inject and clean
new_cols = ['flow_through_ratio', 'aggregation_ratio', 'balance_drain_intensity']

for col in new_cols:
    if col not in master_v38.columns:
        print(f"⚠️ Warning: '{col}' was ghosted. Injecting safe defaults.")
        master_v38[col] = 0.0
    else:
        master_v38[col] = master_v38[col].fillna(0.0)

master_v38_pl = pl.from_pandas(master_v38)
master_v38_pl.write_parquet("/kaggle/working/master_features_v38_WASHPIPE.parquet")

print(f"✅ V38 SECURED! Matrix Shape: {master_v38.shape}")
print("Wash-Pipe (Pass-Through) logic successfully injected.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import gc

print("--- 🛡️ IGNITING LEVEL 39: THE DYNAMIC ENTROPY FORGE ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
cat_results = []

print("[1/3] Sweeping batches for Structural Entropy...")
for i in range(1, 5):
    tx_batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    tx_add_batch_dir = f"{DATA_PATH}/transactions_additional/batch-{i}"
    print(f"  -> Processing Categorical Physics for Batch {i}...")
    
    # 1. Dynamically read the schema so Polars doesn't crash on missing columns
    base_lazy = pl.scan_parquet(f"{tx_batch_dir}/*.parquet")
    base_cols = base_lazy.collect_schema().names()
    
    add_lazy = pl.scan_parquet(f"{tx_add_batch_dir}/*.parquet")
    add_cols = add_lazy.collect_schema().names()
    
    # We ALWAYS have channel in the base transactions
    calc_cols = ["transaction_id", "account_id", "channel"]
    add_calc_cols = ["transaction_id"]
    
    aggs = [
        pl.len().alias("total_txns"),
        pl.col("channel").n_unique().alias("channel_diversity")
    ]
    
    # Dynamically check for MCC
    has_mcc = False
    if "mcc_code" in base_cols:
        calc_cols.append("mcc_code")
        aggs.append(pl.col("mcc_code").n_unique().alias("mcc_diversity"))
        has_mcc = True
    elif "mcc_code" in add_cols:
        add_calc_cols.append("mcc_code")
        aggs.append(pl.col("mcc_code").n_unique().alias("mcc_diversity"))
        has_mcc = True
        
    # Dynamically check for Counterparty
    has_cp = False
    if "counterparty_id" in base_cols:
        calc_cols.append("counterparty_id")
        aggs.append(pl.col("counterparty_id").n_unique().alias("cp_diversity"))
        has_cp = True
    elif "counterparty_id" in add_cols:
        add_calc_cols.append("counterparty_id")
        aggs.append(pl.col("counterparty_id").n_unique().alias("cp_diversity"))
        has_cp = True

    # 2. Build the execution graph safely
    base_lazy = base_lazy.select(calc_cols).drop_nulls()
    
    if len(add_calc_cols) > 1: # If we actually found MCC/CP in the additional files
        add_lazy = add_lazy.select(add_calc_cols).drop_nulls()
        joined_lazy = base_lazy.join(add_lazy, on="transaction_id", how="inner")
    else:
        joined_lazy = base_lazy

    # 3. Execute the aggregation
    batch_features = joined_lazy.group_by("account_id").agg(aggs).collect()
    cat_results.append(batch_features)
    gc.collect()

print("\n[2/3] Consolidating Entropy Features...")

# Prepare the aggregation list based on what we actually found
final_aggs = [
    pl.col("total_txns").sum().alias("total_txns"),
    pl.col("channel_diversity").max().fill_null(1)
]
if has_mcc: final_aggs.append(pl.col("mcc_diversity").max().fill_null(1))
if has_cp: final_aggs.append(pl.col("cp_diversity").max().fill_null(1))

cat_df = pl.concat(cat_results).group_by("account_id").agg(final_aggs).to_pandas()

# Calculate Entropy and Top Share
cat_df['channel_entropy'] = np.log1p(cat_df['channel_diversity'])
cat_df['channel_top_share'] = 1.0 - (cat_df['channel_diversity'] / (cat_df['total_txns'] + 1.0))

if has_mcc:
    cat_df['mcc_entropy'] = np.log1p(cat_df['mcc_diversity'])
    cat_df['mcc_top_share'] = 1.0 - (cat_df['mcc_diversity'] / (cat_df['total_txns'] + 1.0))
if has_cp:
    cat_df['counterparty_entropy'] = np.log1p(cat_df['cp_diversity'])
    cat_df['counterparty_concentration'] = 1.0 - (cat_df['cp_diversity'] / (cat_df['total_txns'] + 1.0))

# Clean raw counts
drop_cols = ['total_txns', 'channel_diversity']
if has_mcc: drop_cols.append('mcc_diversity')
if has_cp: drop_cols.append('cp_diversity')
cat_df.drop(columns=drop_cols, inplace=True)

del cat_results; gc.collect()

print("\n[3/3] Injecting into the God-Tier Matrix...")
master_v38 = pl.read_parquet("/kaggle/working/master_features_v38_WASHPIPE.parquet").to_pandas()

master_v39 = master_v38.merge(cat_df, on="account_id", how="left")

# FAIL-SAFE: Inject and clean dynamically
new_cols = ['channel_entropy', 'channel_top_share']
if has_mcc: new_cols.extend(['mcc_entropy', 'mcc_top_share'])
if has_cp: new_cols.extend(['counterparty_entropy', 'counterparty_concentration'])

for col in new_cols:
    if col not in master_v39.columns:
        print(f"⚠️ Warning: '{col}' was ghosted. Injecting safe defaults.")
        master_v39[col] = 0.0
    else:
        master_v39[col] = master_v39[col].fillna(0.0)

master_v39_pl = pl.from_pandas(master_v39)
master_v39_pl.write_parquet("/kaggle/working/master_features_v39_ENTROPY.parquet")

print(f"✅ V39 SECURED! Matrix Shape: {master_v39.shape}")
print(f"Dynamic features injected successfully. (MCC Found: {has_mcc}, Counterparty Found: {has_cp})")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
import gc

print("--- 🛠️ IGNITING LEVEL 41: THE UNSUPERVISED ANOMALY FORGE ---")

# 1. Load V39
print("[1/3] Loading V39 Matrix...")
master_v39 = pl.read_parquet("/kaggle/working/master_features_v39_ENTROPY.parquet").to_pandas()

# Filter strictly numerical columns for unsupervised AI
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in master_v39.columns if c not in drop_cols]
cat_features = master_v39[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"  -> Scaling {len(num_features)} dimensions for Anomaly Detection...")
X_num = master_v39[num_features].fillna(0).copy()
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_num)

# 2. The Anomaly AI Suite
print("\n[2/3] Unleashing Unsupervised Models...")

# Model 1: Isolation Forest (Finds data points that are easy to separate)
print("  -> Training Isolation Forest...")
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
master_v39['iso_anomaly_score'] = iso.fit_predict(X_scaled)
# Convert -1 (anomaly) and 1 (normal) to an actual risk score
master_v39['iso_anomaly_score'] = iso.score_samples(X_scaled) * -1 

# Model 2: Local Outlier Factor (Finds density-based outliers)
print("  -> Training Local Outlier Factor...")
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, n_jobs=-1)
master_v39['lof_score'] = lof.fit_predict(X_scaled)
master_v39['lof_score'] = lof.negative_outlier_factor_ * -1

# Model 3: K-Means Distance (Finds accounts sitting outside main clusters)
print("  -> Training K-Means Outlier Detection...")
kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)
kmeans.fit(X_scaled)
# Distance to nearest cluster center is the anomaly score
distances = kmeans.transform(X_scaled)
master_v39['kmeans_anomaly_score'] = np.min(distances, axis=1)

# Global Anomaly Score (The Ensemble)
master_v39['global_anomaly_score'] = (
    master_v39['iso_anomaly_score'].rank(pct=True) * 0.4 + 
    master_v39['lof_score'].rank(pct=True) * 0.3 + 
    master_v39['kmeans_anomaly_score'].rank(pct=True) * 0.3
)

del X_num, X_scaled; gc.collect()

# 3. Advanced Combos (From your list)
print("\n[3/3] Forging Advanced Combo Features...")

cols = master_v39.columns

# risk_flow_combo: Global Anomaly * Flow Through Ratio
if 'flow_through_ratio' in cols:
    master_v39['risk_flow_combo'] = master_v39['global_anomaly_score'] * master_v39['flow_through_ratio']

# struct_combo: Aggregation Ratio (Smurfing) * Entropy
if 'aggregation_ratio' in cols and 'channel_entropy' in cols:
    master_v39['structuring_behavior_combo'] = master_v39['aggregation_ratio'] * master_v39['channel_entropy']

# burst_entropy_combo: Max Gap * Minute Entropy
if 'max_gap_hours' in cols and 'minute_entropy' in cols:
    master_v39['burst_entropy_combo'] = master_v39['max_gap_hours'] * master_v39['minute_entropy']

master_v39_pl = pl.from_pandas(master_v39)
master_v39_pl.write_parquet("/kaggle/working/master_features_v41_ANOMALY.parquet")

print(f"✅ V41 SECURED! Matrix Shape: {master_v39.shape}")
print("Isolation Forest, LOF, K-Means, and 3 Advanced Combos are locked in.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import gc

print("--- 🛡️ IGNITING LEVEL 42: EARLY IGNITION (DYNAMIC EDITION) ---")

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

print("[1/3] Dynamically Scanning Account Schema...")
accounts_schema = pl.scan_parquet(f"{DATA_PATH}/accounts.parquet").collect_schema().names()

# 1. Branch Density
if "branch_code" in accounts_schema:
    print("  -> Calculating Branch Density...")
    branch_density = (
        pl.scan_parquet(f"{DATA_PATH}/accounts.parquet")
        .group_by("branch_code")
        .agg(pl.len().alias("branch_tx_density"))
        .collect()
    )
    accounts_df = pl.scan_parquet(f"{DATA_PATH}/accounts.parquet").select(["account_id", "branch_code"]).collect().join(branch_density, on="branch_code", how="left")
else:
    print("  -> branch_code missing. Skipping branch density.")
    accounts_df = pl.scan_parquet(f"{DATA_PATH}/accounts.parquet").select(["account_id"]).collect()
    accounts_df = accounts_df.with_columns(pl.lit(0).alias("branch_tx_density"))

# 2. Account Opening Date (Real or Inferred)
has_open_date = "account_opening_date" in accounts_schema

print(f"\n[2/3] Sweeping batches for Early TX Explosion... (Using Official Open Date: {has_open_date})")

# If no open date, we must infer it from the very first transaction across all batches
if not has_open_date:
    print("  -> Inferring Account Opening Dates from first transaction...")
    first_tx_list = []
    for i in range(1, 5):
        first_tx_list.append(
            pl.scan_parquet(f"{DATA_PATH}/transactions/batch-{i}/*.parquet")
            .select(["account_id", "transaction_timestamp"])
            .drop_nulls()
            .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False).alias("ts"))
            .group_by("account_id").agg(pl.col("ts").min().alias("min_ts")).collect()
        )
    inferred_open_dates = pl.concat(first_tx_list).group_by("account_id").agg(pl.col("min_ts").min().alias("open_date"))
else:
    inferred_open_dates = pl.scan_parquet(f"{DATA_PATH}/accounts.parquet").select(["account_id", "account_opening_date"]).with_columns(
        pl.col("account_opening_date").str.to_datetime(strict=False).alias("open_date")
    ).select(["account_id", "open_date"]).collect()

# Now sweep for the first 15 days of volume
early_results = []
for i in range(1, 5):
    tx_batch_dir = f"{DATA_PATH}/transactions/batch-{i}"
    print(f"  -> Processing Early TX Physics for Batch {i}...")
    
    batch_lazy = pl.scan_parquet(f"{tx_batch_dir}/*.parquet").select([
        "account_id", "transaction_timestamp", "amount"
    ]).with_columns([
        pl.col("transaction_timestamp").str.to_datetime(strict=False).alias("ts")
    ])
    
    joined = batch_lazy.join(inferred_open_dates.lazy(), on="account_id", how="inner")
    
    early_batch = joined.filter(
        pl.col("ts") <= (pl.col("open_date") + pl.duration(days=15))
    ).group_by("account_id").agg([
        pl.len().alias("early_tx_count"),
        pl.col("amount").sum().alias("early_tx_volume")
    ]).collect()
    
    early_results.append(early_batch)
    gc.collect()

print("\n[3/3] Consolidating and Injecting into God-Tier Matrix...")
early_df = pl.concat(early_results).group_by("account_id").agg([
    pl.col("early_tx_count").sum().alias("early_tx_count"),
    pl.col("early_tx_volume").sum().alias("early_tx_volume")
]).to_pandas()
del early_results; gc.collect()

accounts_pd = accounts_df.drop("branch_code", strict=False).to_pandas()

# Load V41 Matrix
master_v41 = pl.read_parquet("/kaggle/working/master_features_v41_ANOMALY.parquet").to_pandas()

# Merge
master_v42 = master_v41.merge(accounts_pd, on="account_id", how="left")
master_v42 = master_v42.merge(early_df, on="account_id", how="left")

# Clean and calculate ratio
master_v42['early_tx_count'] = master_v42['early_tx_count'].fillna(0.0)
master_v42['early_tx_volume'] = master_v42['early_tx_volume'].fillna(0.0)
master_v42['branch_tx_density'] = master_v42['branch_tx_density'].fillna(0.0)

if 'total_tx_count' in master_v42.columns:
    master_v42['early_tx_explosion_ratio'] = master_v42['early_tx_count'] / (master_v42['total_tx_count'] + 1.0)
else:
    master_v42['early_tx_explosion_ratio'] = master_v42['early_tx_count'] / (master_v42['early_tx_count'].max() + 1.0)

master_v42_pl = pl.from_pandas(master_v42)
master_v42_pl.write_parquet("/kaggle/working/master_features_v42_IGNITION.parquet")

print(f"✅ V42 SECURED! Matrix Shape: {master_v42.shape}")
print("Early TX Explosion logic locked in dynamically.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import gc

print("--- ⚔️ IGNITING LEVEL 43: THE MIDNIGHT STRIKE ---")

# 1. Load the 230-Feature God-Matrix
print("[1/4] Loading V42 Ignition Matrix...")
df_master = pl.read_parquet("/kaggle/working/master_features_v42_IGNITION.parquet").to_pandas()
last_preds = pd.read_csv("/kaggle/input/notebooks/tanmayistired/iitd-phase-2/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Extreme Pseudo-Label Anchors (The proven baseline)
print("[2/4] Restoring V14 Pure Anchors...")
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)

del df_master, df_train_orig, pseudo_mules, pseudo_legit; gc.collect()

# 3. Clean and Prep
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"Training strictly on {len(features)} injected weapons...")

df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

X_val_orig = df_train[features].copy()
y_val_orig = df_train['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 4. Train the Multi-Seed Ensembles
print("\n[3/4] Igniting Multi-Seed Tree Ensembles...")

SEEDS = [42, 1337, 2026] 

# Multi-Seed CatBoost
cat_preds_list = []
for seed in SEEDS:
    print(f"  -> Training CatBoost (Seed {seed})...")
    cat = CatBoostClassifier(iterations=3500, learning_rate=0.015, depth=8, eval_metric='AUC', auto_class_weights='Balanced', cat_features=cat_features, early_stopping_rounds=200, verbose=0, random_seed=seed)
    cat.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val))
    cat_preds_list.append(cat.predict_proba(df_test_orig[features])[:, 1])
    del cat; gc.collect()
cat_probs_final = np.mean(cat_preds_list, axis=0)

# Multi-Seed XGBoost
xgb_preds_list = []
for seed in SEEDS:
    print(f"  -> Training XGBoost (Seed {seed})...")
    xgb_model = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.012, max_depth=7, scale_pos_weight=12, tree_method='hist', enable_categorical=True, early_stopping_rounds=200, eval_metric='auc', random_state=seed)
    xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=0)
    xgb_preds_list.append(xgb_model.predict_proba(df_test_orig[features])[:, 1])
    del xgb_model; gc.collect()
xgb_probs_final = np.mean(xgb_preds_list, axis=0)

# Multi-Seed LightGBM
lgb_preds_list = []
X_train_lgb = df_train[features].copy()
X_test_lgb = df_test_orig[features].copy()
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

for seed in SEEDS:
    print(f"  -> Training LightGBM (Seed {seed})...")
    lgbm = lgb.LGBMClassifier(n_estimators=3500, learning_rate=0.01, num_leaves=63, max_depth=8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    lgbm.fit(X_train_lgb, df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], eval_metric='auc', callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)])
    lgb_preds_list.append(lgbm.predict_proba(X_test_lgb)[:, 1])
    del lgbm; gc.collect()
lgb_probs_final = np.mean(lgb_preds_list, axis=0)
del X_train_lgb, X_test_lgb; gc.collect()

# 5. The Rank Ensemble
print("\n[4/4] Executing Multi-Seed Rank Ensembling...")
cat_ranks = pd.Series(cat_probs_final).rank(pct=True).values
xgb_ranks = pd.Series(xgb_probs_final).rank(pct=True).values
lgb_ranks = pd.Series(lgb_probs_final).rank(pct=True).values

# 40% CatBoost | 30% LightGBM | 30% XGBoost
final_ranks = (cat_ranks * 0.40) + (lgb_ranks * 0.30) + (xgb_ranks * 0.30)
sub_df = pd.DataFrame({'account_id': df_test_orig['account_id'].values, 'is_mule': final_ranks})

# 6. Temporal Sniper
mule_accounts = sub_df[sub_df['is_mule'] > 0.85]['account_id'].dropna().astype(str).tolist()

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False).dt.timestamp("ms").alias("ts_ms"))
    .group_by("account_id")
    .agg([
        pl.col("ts_ms").quantile(0.05).cast(pl.Datetime(time_unit="ms")).alias("start_ts"),
        pl.col("ts_ms").quantile(0.95).cast(pl.Datetime(time_unit="ms")).alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V43_MIDNIGHT_STRIKE.csv", index=False)

print("🚀 DONE. V43_MIDNIGHT_STRIKE.csv IS READY. DROP IT ON THE PORTAL.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import gc
import matplotlib.pyplot as plt

print("--- 🪓 IGNITING LEVEL 44: THE SHAP PRUNING PROTOCOL ---")

# 1. Load V42 from the updated Input Path
V42_PATH = "/kaggle/input/notebooks/tanmayistired/iitd-phase-2/master_features_v42_IGNITION.parquet"
print(f"[1/4] Loading 230-Feature Matrix from: {V42_PATH}")

df_master = pl.read_parquet(V42_PATH).to_pandas()

# Filter strictly to the training data (we need ground truth labels to calculate SHAP)
df_train = df_master[df_master['is_mule'].notnull()].copy()

# 2. Clean and Prep
drop_cols = [
    'account_id', 'is_mule', 'customer_id', 'account_opening_date', 
    'last_mobile_update_date', 'date_of_birth', 'relationship_start_date',
    'freeze_date', 'unfreeze_date', 'account_status', 'last_kyc_date', 
    'passbook_last_update_date', 'name', 'branch_address', 'branch_pin', 
    'customer_pin', 'branch_code', 'cust_region', 'branch_region', 
    'branch_city', 'branch_state', 'gender', 'address'
]

features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

df_train[num_features] = df_train[num_features].fillna(-1)
for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')

print(f"[2/4] Igniting Evaluator Model on {len(features)} features...")

# 3. Train the Evaluator Model
X_train = df_train[features]
y_train = df_train['is_mule'].astype(int)

# Fast LGBM purely to extract feature weights
eval_model = lgb.LGBMClassifier(
    n_estimators=500, 
    learning_rate=0.05, 
    num_leaves=31, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1,
    verbose=-1
)
eval_model.fit(X_train, y_train)

print("\n[3/4] Executing Game-Theoretic SHAP Analysis (This takes a moment)...")
# Calculate SHAP values (Using TreeExplainer for speed)
explainer = shap.TreeExplainer(eval_model)

# SHAP on a random sample to save RAM and Time
X_sample = X_train.sample(n=15000, random_state=42)
shap_values = explainer.shap_values(X_sample)

# If binary classification, shap_values might be a list. We want the absolute mean SHAP for the positive class.
if isinstance(shap_values, list):
    shap_abs = np.abs(shap_values[1]).mean(axis=0)
else:
    shap_abs = np.abs(shap_values).mean(axis=0)

# Create Importance DataFrame
importance_df = pd.DataFrame({
    'feature': features,
    'shap_importance': shap_abs
}).sort_values(by='shap_importance', ascending=False)

print("\n🔥 THE TOP 15 MOST LETHAL FEATURES 🔥")
print(importance_df.head(15).to_string(index=False))

# 4. The Pruning (Keep Top 60)
TOP_K = 60
print(f"\n[4/4] Pruning Matrix... Slicing Top {TOP_K} Features.")

top_features = importance_df['feature'].head(TOP_K).tolist()
final_keep_cols = ['account_id', 'is_mule'] + top_features

master_v43_pruned = df_master[final_keep_cols].copy()

# Save the hyper-focused matrix to the new working directory
master_v43_pl = pl.from_pandas(master_v43_pruned)
master_v43_pl.write_parquet("/kaggle/working/master_features_v43_PRUNED.parquet")

print(f"✅ V43 PRUNED SECURED! New Matrix Shape: {master_v43_pruned.shape}")
print("Dead weight dropped. Ready for the next phase.")

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score

print("--- IGNITING LEVEL 45: THE LEAK HUNT (DARK ARTS) ---")

print("[1/3] Loading Raw Infrastructure...")
DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"

labels = pl.read_parquet(f"{DATA_PATH}/train_labels.parquet").to_pandas()
accounts = pl.read_parquet(f"{DATA_PATH}/accounts.parquet").to_pandas()

df = accounts.merge(labels, on='account_id', how='left')
df_train = df[df['is_mule'].notnull()].copy()

print("\n[2/3] Executing Leak Test 1: ID Sequence Analysis...")
# Rank the account IDs to see if mules were generated sequentially
df_train['id_rank'] = df_train['account_id'].rank()

id_auc = roc_auc_score(df_train['is_mule'], df_train['id_rank'])
if id_auc < 0.5:
    id_auc = 1.0 - id_auc
    
print(f"  -> AUC of predicting mule using ONLY Account ID sorting: {id_auc:.4f}")
if id_auc > 0.60:
    print("  -> CRITICAL WARNING: The RBIH generated Mule IDs sequentially. Massive Data Leak detected.")
else:
    print("  -> ID sequences look clean. No obvious leak here.")

print("\n[3/3] Executing Leak Test 2: The Null Exploit...")
# Check if the sheer volume of missing data is a perfect predictor
df_train['missing_count'] = df_train.isnull().sum(axis=1)

null_auc = roc_auc_score(df_train['is_mule'], df_train['missing_count'])
if null_auc < 0.5:
    null_auc = 1.0 - null_auc
    
print(f"  -> AUC of predicting mule using ONLY the number of missing columns: {null_auc:.4f}")
if null_auc > 0.65:
    print("  -> CRITICAL WARNING: Mules have structurally different null patterns. Data Leak detected.")
else:
    print("  -> Null distributions look standard.")
    
print("\nLeak scan complete.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
import gc

print("--- 🎯 IGNITING LEVEL 46: THE F1 THRESHOLD HACKER ---")

# 1. Load the Pruned God-Matrix
V43_PATH = "/kaggle/working/master_features_v43_PRUNED.parquet"
print(f"[1/3] Loading Pruned V43 Matrix from: {V43_PATH}")

df_master = pl.read_parquet(V43_PATH).to_pandas()
df_train = df_master[df_master['is_mule'].notnull()].copy().reset_index(drop=True)

drop_cols = ['account_id', 'is_mule']
features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

df_train[num_features] = df_train[num_features].fillna(-1)
for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')

X = df_train[features]
y = df_train['is_mule'].astype(int)

# 2. Out-of-Fold Predictions (To prevent overfitting the threshold)
print("\n[2/3] Generating Out-Of-Fold Probabilities...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(df_train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.02, num_leaves=31, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    
    oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]

# 3. The Threshold Sweep
print("\n[3/3] Sweeping Decimals for Maximum F1 Score...")
thresholds = np.arange(0.01, 1.00, 0.01)
best_f1 = 0
best_thresh = 0
best_prec = 0
best_rec = 0

for thresh in thresholds:
    # Convert probabilities to hard 0 or 1 based on the threshold
    hard_preds = (oof_preds >= thresh).astype(int)
    
    current_f1 = f1_score(y, hard_preds)
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_thresh = thresh
        best_prec = precision_score(y, hard_preds)
        best_rec = recall_score(y, hard_preds)

print("\n" + "="*50)
print("🚨 OPTIMAL LEADERBOARD CALIBRATION FOUND 🚨")
print("="*50)
print(f"Optimal F1 Threshold : {best_thresh:.2f}")
print(f"Maximum F1 Score     : {best_f1:.4f}")
print(f"Precision at Peak    : {best_prec:.4f}")
print(f"Recall at Peak       : {best_rec:.4f}")
print("="*50)

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import gc

print("--- IGNITING LEVEL 47: THE PIECEWISE CALIBRATOR (FINAL STRIKE) ---")

# 1. Load the Pruned Matrix
print("[1/4] Loading V43 Pruned Matrix...")
V43_PATH = "/kaggle/working/master_features_v43_PRUNED.parquet"
df_master = pl.read_parquet(V43_PATH).to_pandas()
last_preds = pd.read_csv("/kaggle/input/datasets/tanmayistired/v14-final-push-something/DONE_WITHIT_V14_FINAL_PUSH.csv")

df_train_orig = df_master[df_master['is_mule'].notnull()].copy()
df_test_orig = df_master[df_master['is_mule'].isnull()].copy()

# 2. Extreme Pseudo-Label Anchors
print("[2/4] Restoring V14 Pure Anchors...")
confident_mules = last_preds[last_preds['is_mule'] > 0.999]['account_id']
confident_legit = last_preds[last_preds['is_mule'] < 0.001]['account_id']

pseudo_mules = df_test_orig[df_test_orig['account_id'].isin(confident_mules)].copy()
pseudo_mules['is_mule'] = 1.0
pseudo_legit = df_test_orig[df_test_orig['account_id'].isin(confident_legit)].copy()
pseudo_legit['is_mule'] = 0.0

df_train = pd.concat([df_train_orig, pseudo_mules, pseudo_legit], axis=0).reset_index(drop=True)

del df_master, df_train_orig, pseudo_mules, pseudo_legit; gc.collect()

# 3. Clean and Prep
drop_cols = ['account_id', 'is_mule']
features = [c for c in df_train.columns if c not in drop_cols]
cat_features = df_train[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
num_features = [c for c in features if c not in cat_features]

print(f"Training strictly on {len(features)} pruned, high-signal features...")

df_train[num_features] = df_train[num_features].fillna(-1)
df_test_orig[num_features] = df_test_orig[num_features].fillna(-1)

for c in cat_features:
    df_train[c] = df_train[c].fillna("UNKNOWN").astype(str).astype('category')
    df_test_orig[c] = df_test_orig[c].fillna("UNKNOWN").astype(str).astype('category')

X_val_orig = df_train[features].copy()
y_val_orig = df_train['is_mule'].astype(int)
_, X_val, _, y_val = train_test_split(X_val_orig, y_val_orig, test_size=0.15, stratify=y_val_orig, random_state=42)

# 4. Train the Multi-Seed Ensembles
print("\n[3/4] Igniting Multi-Seed Tree Ensembles...")

SEEDS = [42, 1337, 2026]

# Multi-Seed CatBoost
cat_preds_list = []
for seed in SEEDS:
    cat = CatBoostClassifier(iterations=3500, learning_rate=0.015, depth=8, eval_metric='AUC', auto_class_weights='Balanced', cat_features=cat_features, early_stopping_rounds=200, verbose=0, random_seed=seed)
    cat.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=(X_val, y_val))
    cat_preds_list.append(cat.predict_proba(df_test_orig[features])[:, 1])
    del cat; gc.collect()
cat_probs_final = np.mean(cat_preds_list, axis=0)

# Multi-Seed XGBoost
xgb_preds_list = []
for seed in SEEDS:
    xgb_model = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.012, max_depth=7, scale_pos_weight=12, tree_method='hist', enable_categorical=True, early_stopping_rounds=200, eval_metric='auc', random_state=seed)
    xgb_model.fit(df_train[features], df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], verbose=0)
    xgb_preds_list.append(xgb_model.predict_proba(df_test_orig[features])[:, 1])
    del xgb_model; gc.collect()
xgb_probs_final = np.mean(xgb_preds_list, axis=0)

# Multi-Seed LightGBM
lgb_preds_list = []
X_train_lgb = df_train[features].copy()
X_test_lgb = df_test_orig[features].copy()
for c in cat_features:
    X_train_lgb[c] = X_train_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

for seed in SEEDS:
    lgbm = lgb.LGBMClassifier(n_estimators=3500, learning_rate=0.01, num_leaves=63, max_depth=8, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    lgbm.fit(X_train_lgb, df_train['is_mule'].astype(int), eval_set=[(X_val, y_val)], eval_metric='auc', callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)])
    lgb_preds_list.append(lgbm.predict_proba(X_test_lgb)[:, 1])
    del lgbm; gc.collect()
lgb_probs_final = np.mean(lgb_preds_list, axis=0)
del X_train_lgb, X_test_lgb; gc.collect()

# 5. Piecewise Threshold Calibration
print("\n[4/4] Executing Piecewise F1 Calibration...")
# Ensembling raw probabilities, not ranks
raw_probs = (cat_probs_final * 0.40) + (lgb_probs_final * 0.30) + (xgb_probs_final * 0.30)

# The mathematical shift: map 0.71 to 0.50
OPTIMAL_THRESH = 0.71
calibrated_probs = np.where(
    raw_probs < OPTIMAL_THRESH,
    raw_probs * (0.50 / OPTIMAL_THRESH),
    0.50 + ((raw_probs - OPTIMAL_THRESH) * (0.50 / (1.0 - OPTIMAL_THRESH)))
)

sub_df = pd.DataFrame({'account_id': df_test_orig['account_id'].values, 'is_mule': calibrated_probs})

# 6. Temporal Sniper
mule_accounts = sub_df[sub_df['is_mule'] > 0.85]['account_id'].dropna().astype(str).tolist()

DATA_PATH = "/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2"
mule_times = (
    pl.scan_parquet(f"{DATA_PATH}/transactions/batch-*/*.parquet")
    .select(["account_id", "transaction_timestamp"])
    .filter(pl.col("account_id").is_in(mule_accounts))
    .with_columns(pl.col("transaction_timestamp").str.to_datetime(strict=False).dt.timestamp("ms").alias("ts_ms"))
    .group_by("account_id")
    .agg([
        pl.col("ts_ms").quantile(0.05).cast(pl.Datetime(time_unit="ms")).alias("start_ts"),
        pl.col("ts_ms").quantile(0.95).cast(pl.Datetime(time_unit="ms")).alias("end_ts")
    ]).collect()
).to_pandas()

mule_times['suspicious_start'] = pd.to_datetime(mule_times['start_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")
mule_times['suspicious_end'] = pd.to_datetime(mule_times['end_ts']).dt.strftime("%Y-%m-%dT%H:%M:%S")

final_sub = sub_df.merge(mule_times[['account_id', 'suspicious_start', 'suspicious_end']], on='account_id', how='left').fillna('')
final_sub.to_csv("/kaggle/working/V47_CALIBRATED_STRIKE.csv", index=False)

print("DONE. V47_CALIBRATED_STRIKE.csv IS READY.")